# Coordinate robustness: updated Colab analysis

This companion addresses the reviewer's coordinate and laboratory-specificity comment. It writes a **new resumable experiment** and reads your previous training experiment without modifying it. It never loads external data.

**Default run:** AP-left, SUB01, all seven model families requested. The six completed original models can be reused; Conv1D-LSTM is explicitly unavailable in the fast mode until a valid original selection/refit exists.

1. Evaluate constant translations using the saved absolute-coordinate models. This is inference only.
2. Refit models using translation-normalized trajectories, then compare the same held-out cycles and repeat the perturbations.
3. Run pelvis-centered, gait-direction-aligned, body-scale-normalized and combined representations **when the required verified inputs are supplied**. Otherwise record a specific blocked status.
4. Run rotation tests only with verified 3D ankle/direction data and a compatible force-coordinate definition.

Choose a **GPU runtime** for the refitting cells, then Run all and mount the same Drive account. Package versions are taken from your saved experiment. If setup requests a runtime restart after installing packages, restart once and Run all again.

**Two analysis strategies**

- `frozen_selection` (default): use the architecture-specific hyperparameters and epoch count already selected by inner LOSO on absolute coordinates, then refit weights on each new representation. This is a focused coordinate sensitivity analysis, with much less fitting than a new search. It does **not** reselect hyperparameters for normalized coordinates and must not be called a complete nested rerun of each representation.
- `nested_reselect`: repeat complete subject-wise inner LOSO selection for each representation and outer fold before refitting. Uses the original search space, trial budget and epoch limit. This is substantially more expensive. Set `TASK_SELECTION='all'` and `OUTER_SELECTION='all'` for the complete available internal-cohort repeat; missing anatomical inputs still remain blocked.

SUB01 has already been examined. This follow-up is exploratory: do not optimize settings or perturbation magnitudes using its test results. Neither a default one-participant run nor preprocessing invariance proves biomechanical causality.

In [ ]:
from pathlib import Path
import json, sys, re, subprocess, importlib.metadata
from google.colab import drive
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/Shida_Nested_LOSO')
SOURCE_EXPERIMENT = 'reviewer_complete_v1'
RUN_ID = 'coordinate_checks_v1'
TASK_SELECTION = 'AP_Left'       # 'all' or comma-separated task names
OUTER_SELECTION = 'SUB01'       # 'all' or comma-separated participant IDs
STRATEGY = 'frozen_selection'   # 'frozen_selection' or 'nested_reselect'
REQUESTED_MODES = ['absolute','translation','pelvis','aligned','body','combined']
REQUESTED_MODELS = ['mean_waveform','ridge','phase_only','shallow_mlp','lstm','gru','conv1d_lstm']
RUN_TRANSLATION_TESTS = True
TRANSLATION_SD_MULTIPLIERS = [-1.0,-0.5,-0.25,0.25,0.5,1.0]
RUN_ROTATION_TESTS = True
ROTATION_ANGLES_DEG = [-15.0,15.0]
INSTALL_MATCHING_PACKAGES = True
CONFIRMED_ANKLE_UNITS = 'unverified'  # Set to 'mm' or 'm' only after checking the source.
ANKLE_UNITS_EVIDENCE = ''             # Document the source supporting that unit.

for label,value in [('SOURCE_EXPERIMENT',SOURCE_EXPERIMENT),('RUN_ID',RUN_ID)]:
    if not re.fullmatch(r'[A-Za-z0-9_-]+',value):raise ValueError(f'Invalid {label}')
if STRATEGY not in ('frozen_selection','nested_reselect'):raise ValueError('Unknown STRATEGY')
if REQUESTED_MODES[0]!='absolute':raise ValueError('Keep absolute first to obtain saved-model perturbation results before long refits')
if len(set(REQUESTED_MODES))!=len(REQUESTED_MODES) or len(set(REQUESTED_MODELS))!=len(REQUESTED_MODELS):
    raise ValueError('Duplicate mode/model')
if not TRANSLATION_SD_MULTIPLIERS or any(not isinstance(x,(int,float)) or x==0 or abs(x)>100 for x in TRANSLATION_SD_MULTIPLIERS):
    raise ValueError('Use a nonempty, finite prespecified list of nonzero SD multipliers')
DATA_ZIP = PROJECT/'datasets/Shida-LOSO-ANKLE.zip'
SOURCE_ROOT = PROJECT/'complete_studies'/SOURCE_EXPERIMENT
REFERENCE_SPEC = SOURCE_ROOT/'AP_Left/absolute/prespecification.json'
OUTPUT = PROJECT/'coordinate_robustness'/RUN_ID/STRATEGY
SUPPLEMENT = PROJECT/'coordinate_inputs'
CODE = Path('/content/shida_coordinate_robustness_code')
if not DATA_ZIP.is_file() or not REFERENCE_SPEC.is_file():
    raise FileNotFoundError('Original ZIP or AP_Left absolute prespecification missing. Check the same Drive account and paths.')
reference = json.loads(REFERENCE_SPEC.read_text())
print('Original training data and models:',SOURCE_ROOT)
print('New results and checkpoints:',OUTPUT)
print('Optional anatomical data:',SUPPLEMENT)
print('Strategy:',STRATEGY)

## Match the original software

This cell installs only missing/mismatched recorded packages. It does not change the original model files. Original data/preprocessing modules are embedded unchanged, and their hashes are checked against existing experiments.

New searches use a separate copy of the training module with one declared change: the consecutive-failure threshold is set above the total trial budget, allowing later candidates to run after numerical failures. The search space, objective and trial budget remain unchanged. A candidate with any failed inner participant is not scored from its remaining folds. A failed model/coordinate case is logged and does not stop unrelated cases.

In [ ]:
pins=[]
for package in ['numpy','pandas','scikit-learn','tensorflow','keras','keras-tuner','joblib']:
    wanted=reference['software'].get(package)
    if not wanted or not re.fullmatch(r'[0-9][A-Za-z0-9.+_-]*',wanted):
        raise ValueError(f'No valid original version recorded for {package}')
    try:current=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:current=None
    if current!=wanted:pins.append(f'{package}=={wanted}')
for package in ['openpyxl','matplotlib']:
    try:importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:pins.append(package)
if pins:
    print('Required packages:',pins)
    if not INSTALL_MATCHING_PACKAGES:raise RuntimeError('Enable INSTALL_MATCHING_PACKAGES or install the displayed versions.')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*pins])
module_names={'scikit-learn':'sklearn','keras-tuner':'keras_tuner'}
stale=[]
for package,wanted in reference['software'].items():
    module=sys.modules.get(module_names.get(package,package))
    if module is not None and getattr(module,'__version__',wanted)!=wanted:stale.append(package)
if stale:
    raise RuntimeError('Packages were installed but old modules remain loaded: '+', '.join(stale)+
                       '. Select Runtime > Restart session once, then Run all again.')
print('Package setup ready. Recorded training Python:',reference['software'].get('python'))

In [ ]:
SOURCES = {'data.py': '"""Read the paired per-participant Raw workbooks in Shida-LOSO-ANKLE."""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom hashlib import sha256\nfrom io import BytesIO\nfrom pathlib import Path\nimport re\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport pandas as pd\n\n\nTASKS = {\n    "apleft": "AP_Left", "apright": "AP_Right",\n    "longlef": "Vertical_Left", "longleft": "Vertical_Left",\n    "longrig": "Vertical_Right", "longright": "Vertical_Right",\n    "verticalleft": "Vertical_Left", "verticalright": "Vertical_Right",\n    "mlleft": "ML_Left", "mlright": "ML_Right",\n}\nTASK_ORDER = ["AP_Left", "AP_Right", "Vertical_Left", "Vertical_Right", "ML_Left", "ML_Right"]\nRAW_PATTERN = re.compile(r"(?:^|/)Raw/([^/]+)/([^/]+\\.xlsx)$", re.I)\nSUBJECT_PATTERN = re.compile(r"(?:SUB|UB)(\\d+)", re.I)\n\n\n@dataclass\nclass Dataset:\n    task: str\n    x: np.ndarray                       # [cycles, 100 time points, 1 feature]\n    y: np.ndarray\n    groups: np.ndarray                  # participant ID for EVERY cycle\n    cycles: pd.DataFrame                # source and cycle-position provenance\n    fingerprint: str\n\n    @property\n    def subjects(self):\n        return sorted(np.unique(self.groups).tolist())\n\n    def subset(self, subjects):\n        mask = np.isin(self.groups, subjects)\n        if not mask.any():\n            raise ValueError("Empty participant subset")\n        fingerprint = sha256((self.fingerprint + repr(sorted(subjects))).encode()).hexdigest()\n        return Dataset(self.task, self.x[mask], self.y[mask], self.groups[mask],\n                       self.cycles.loc[mask].reset_index(drop=True), fingerprint)\n\n\ndef canonical_subject(text):\n    matches = SUBJECT_PATTERN.findall(str(text))\n    ids = {f"SUB{int(value):02d}" for value in matches}\n    if len(ids) != 1:\n        raise ValueError(f"Expected one unambiguous participant in {text!r}; got {ids}")\n    return ids.pop()\n\n\ndef workbook_arrays(payload, filename, subject, expected_timesteps=100):\n    """Keep the original in/out column order. GRF headers lack trial identifiers."""\n    with pd.ExcelFile(BytesIO(payload), engine="openpyxl") as book:\n        if not {"in", "out"}.issubset(book.sheet_names):\n            raise ValueError(f"{filename}: paired \'in\' and \'out\' sheets are required")\n        inputs = pd.read_excel(book, sheet_name="in", header=0)\n        targets = pd.read_excel(book, sheet_name="out", header=0)\n    if inputs.shape != targets.shape or inputs.shape[0] != expected_timesteps or not inputs.shape[1]:\n        raise ValueError(f"{filename}: expected matched ({expected_timesteps}, cycles) sheets; "\n                         f"got in={inputs.shape}, out={targets.shape}")\n    for header in inputs.columns:\n        if canonical_subject(header) != subject:\n            raise ValueError(f"{filename}: input column belongs to a different participant: {header}")\n    try:\n        x = inputs.to_numpy(dtype=np.float32).T[:, :, None]\n        y = targets.to_numpy(dtype=np.float32).T[:, :, None]\n    except (ValueError, TypeError) as exc:\n        raise ValueError(f"{filename}: nonnumeric waveform value") from exc\n    if not np.isfinite(x).all() or not np.isfinite(y).all():\n        raise ValueError(f"{filename}: missing/nonfinite values; no silent zero filling or exclusion is allowed")\n    metadata = [{"subject": subject, "cycle_id": f"{subject}:C{i+1:03d}",\n                 "column_position_1based": i + 1, "input_header": str(inputs.columns[i]),\n                 "target_header": str(targets.columns[i]), "source_workbook": filename}\n                for i in range(len(x))]\n    return x, y, metadata\n\n\ndef load_datasets(source, requested_tasks=None):\n    """Accept the uploaded ZIP, its extracted root, or the Raw directory itself.\n\n    Never load the preassembled Train files: they repeat subjects across folds\n    and some supplied input matrices were concatenated along the wrong axis.\n    """\n    source = Path(source).expanduser().resolve()\n    requested = set(requested_tasks or TASK_ORDER)\n    unknown = requested - set(TASK_ORDER)\n    if unknown:\n        raise ValueError(f"Unknown tasks: {sorted(unknown)}")\n    archive = None\n    if source.is_file() and source.suffix.lower() == ".zip":\n        archive = ZipFile(source)\n        names = sorted(n for n in archive.namelist() if RAW_PATTERN.search(n))\n        reader = archive.read\n    elif source.is_dir():\n        paths = {p.as_posix(): p for p in source.rglob("*.xlsx") if RAW_PATTERN.search(p.as_posix())}\n        names = sorted(paths)\n        reader = lambda name: paths[name].read_bytes()\n    else:\n        raise ValueError(f"Expected an existing ZIP or directory: {source}")\n    accum, seen, fingerprints = {}, set(), {}\n    try:\n        for name in names:\n            match = RAW_PATTERN.search(name)\n            task = TASKS.get(re.sub(r"[^a-z]", "", match.group(1).lower()))\n            if task not in requested:\n                continue\n            subject = canonical_subject(match.group(2))\n            if (task, subject) in seen:\n                raise ValueError(f"Duplicate Raw workbook for {task}/{subject}; provide one dataset root")\n            seen.add((task, subject))\n            payload = reader(name)\n            relative_name = "Raw/" + name.split("/")[-2] + "/" + name.split("/")[-1]\n            x, y, metadata = workbook_arrays(payload, relative_name, subject)\n            accum.setdefault(task, []).append((subject, x, y, metadata))\n            fingerprints.setdefault(task, sha256()).update(relative_name.encode() + sha256(payload).digest())\n    finally:\n        if archive is not None:\n            archive.close()\n    absent = requested - set(accum)\n    if absent:\n        raise ValueError(f"No Raw workbooks found for {sorted(absent)} in {source}")\n    result = {}\n    for task in TASK_ORDER:\n        if task not in accum:\n            continue\n        records = sorted(accum[task], key=lambda item: item[0])\n        if len(records) < 3:\n            raise ValueError(f"{task}: at least three participants are required for nested LOSO")\n        result[task] = Dataset(\n            task, np.concatenate([r[1] for r in records]), np.concatenate([r[2] for r in records]),\n            np.concatenate([np.repeat(r[0], len(r[1])) for r in records]),\n            pd.DataFrame([m for r in records for m in r[3]]), fingerprints[task].hexdigest())\n    return result\n\n\ndef nested_indices(groups, outer_subject):\n    """Yield train, validation, test indices; one entire subject in each holdout."""\n    groups = np.asarray(groups)\n    subjects = sorted(np.unique(groups).tolist())\n    if outer_subject not in subjects or len(subjects) < 3:\n        raise ValueError("Unknown outer participant or fewer than three participants")\n    test = np.flatnonzero(groups == outer_subject)\n    for validation_subject in subjects:\n        if validation_subject == outer_subject:\n            continue\n        validation = np.flatnonzero(groups == validation_subject)\n        train = np.flatnonzero((groups != outer_subject) & (groups != validation_subject))\n        assert_subject_separation(groups, train, validation, test)\n        yield validation_subject, train, validation, test\n\n\ndef assert_subject_separation(groups, train, validation, test):\n    combined = np.concatenate([train, validation, test])\n    if not np.array_equal(np.sort(combined), np.arange(len(groups))):\n        raise ValueError("Each cycle must appear exactly once across the three partitions")\n    sets = [set(np.asarray(groups)[idx]) for idx in [train, validation, test]]\n    if any(not s for s in sets) or sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:\n        raise ValueError("Participant overlap or an empty train/validation/test partition")\n\n\ndef audit_datasets(datasets):\n    tasks, participants, cycles, splits = [], [], [], []\n    for task, ds in datasets.items():\n        tasks.append({"task": task, "n_participants": len(ds.subjects), "n_cycles": len(ds.x),\n                      "timepoints": ds.x.shape[1], "features": ds.x.shape[2],\n                      "missing_input_values": int((~np.isfinite(ds.x)).sum()),\n                      "missing_target_values": int((~np.isfinite(ds.y)).sum()),\n                      "subjects": ds.subjects, "fingerprint": ds.fingerprint})\n        cycles.extend(ds.cycles.assign(task=task).to_dict("records"))\n        for subject in ds.subjects:\n            participants.append({"task": task, "subject": subject,\n                                 "n_cycles": int(np.sum(ds.groups == subject))})\n            for val_subject, tr, va, te in nested_indices(ds.groups, subject):\n                splits.append({"task": task, "outer_test_subject": subject,\n                               "inner_validation_subject": val_subject,\n                               "inner_training_subjects": sorted(set(ds.groups[tr])),\n                               "n_train_cycles": len(tr), "n_validation_cycles": len(va),\n                               "n_test_cycles": len(te)})\n    return {"tasks": tasks, "participants": participants, "cycles": cycles, "splits": splits}\n', 'protocol.py': '"""Backend-independent nested LOSO protocol, metrics, and resumable records."""\nfrom __future__ import annotations\n\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nfrom data import Dataset, nested_indices\n\n\ndef write_json(path, obj):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(obj, indent=2, sort_keys=True, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef stable_seed(seed, *labels):\n    text = "|".join(map(str, (seed, *labels)))\n    return int.from_bytes(sha256(text.encode()).digest()[:4], "big") % (2**31 - 1)\n\n\ndef parameter_hash(parameters):\n    return sha256(json.dumps(parameters, sort_keys=True).encode()).hexdigest()\n\n\ndef scale_from_training(train, *others, enabled=False):\n    """Only training inputs determine scale. Targets retain their original units."""\n    if enabled:\n        mean = np.mean(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.std(train, axis=(0, 1), keepdims=True, dtype=np.float64)\n        std = np.where(std < 1e-8, 1.0, std)\n    else:\n        mean = np.zeros((1, 1, train.shape[-1]))\n        std = np.ones_like(mean)\n    arrays = [((arr - mean) / std).astype(np.float32) for arr in (train,) + others]\n    return arrays, {"enabled": enabled, "mean": mean.reshape(-1).tolist(),\n                    "std": std.reshape(-1).tolist()}\n\n\ndef cycle_metrics(truth, prediction):\n    if truth.shape != prediction.shape or not np.isfinite(prediction).all():\n        raise ValueError("Invalid prediction shape or nonfinite predictions")\n    error = truth.astype(np.float64) - prediction.astype(np.float64)\n    rmse = np.sqrt(np.mean(error**2, axis=(1, 2)))\n    mae = np.mean(np.abs(error), axis=(1, 2))\n    span = np.ptp(truth.astype(np.float64), axis=1).reshape(-1)\n    nrmse = np.divide(rmse, span, out=np.full_like(rmse, np.nan), where=span > 1e-12)\n    return pd.DataFrame({"rmse": rmse, "nrmse": nrmse, "mae": mae,\n                         "target_range": span, "nrmse_defined": span > 1e-12})\n\n\ndef evaluate_candidate(ds, outer_subject, parameters, backend, seed, directory, scale_input=False):\n    """Score one candidate equally across inner validation participants.\n\n    The backend only receives inner train/validation data. Outer test arrays\n    are never supplied to tuning, early stopping, or input scaling.\n    """\n    directory = Path(directory)\n    signature = parameter_hash(parameters)\n    records = []\n    for validation_subject, tr, va, _ in nested_indices(ds.groups, outer_subject):\n        path = directory / f"inner_{validation_subject}.json"\n        if path.exists():\n            record = read_json(path)\n            if record["parameter_hash"] != signature:\n                raise ValueError(f"Cached trial has different parameters: {path}")\n            records.append(record)\n            continue\n        (xtr, xva), scaler = scale_from_training(ds.x[tr], ds.x[va], enabled=scale_input)\n        fit_seed = stable_seed(seed, ds.task, outer_subject, validation_subject, "inner")\n        fit = backend.fit_inner(xtr, ds.y[tr], xva, ds.y[va], parameters, fit_seed,\n                                directory / f"history_{validation_subject}.csv")\n        metric = cycle_metrics(ds.y[va], fit["prediction"])\n        record = {"outer_subject": outer_subject, "validation_subject": validation_subject,\n                  "training_subjects": sorted(set(ds.groups[tr])), "n_training_cycles": len(tr),\n                  "n_validation_cycles": len(va), "seed": fit_seed,\n                  "parameter_hash": signature, "scaler": scaler,\n                  "best_epoch": int(fit["best_epoch"]), "epochs_run": int(fit["epochs_run"]),\n                  "validation_mae": float(metric.mae.mean()),\n                  "validation_rmse": float(metric.rmse.mean())}\n        write_json(path, record)\n        records.append(record)\n    score = float(np.mean([r["validation_mae"] for r in records]))\n    if not np.isfinite(score):\n        raise ValueError("Nonfinite inner objective")\n    summary = {"parameters": parameters, "inner_subject_mae": score, "folds": records}\n    write_json(directory / "inner_summary.json", summary)\n    return summary\n\n\ndef refit_and_evaluate(ds, outer_subject, parameters, inner_summary, backend, seed,\n                       directory, architecture, scale_input=False):\n    """Refit on ALL outer-training participants for an inner-derived epoch budget."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    tr = np.flatnonzero(ds.groups != outer_subject)\n    te = np.flatnonzero(ds.groups == outer_subject)\n    if len(te) == 0 or set(ds.groups[tr]) & set(ds.groups[te]):\n        raise ValueError("Invalid outer subject separation")\n    if set(r["validation_subject"] for r in inner_summary["folds"]) != set(ds.groups[tr]):\n        raise ValueError("The winning trial did not validate every outer-training participant")\n    if any(r["outer_subject"] != outer_subject for r in inner_summary["folds"]):\n        raise ValueError("Inner summary belongs to a different outer fold")\n    if parameter_hash(inner_summary["parameters"]) != parameter_hash(parameters):\n        raise ValueError("Refit parameters differ from the selected inner trial")\n    # Predefined rule: ceiling of median inner best epoch; outer test never selects epochs.\n    epochs = max(1, int(np.ceil(np.median([r["best_epoch"] for r in inner_summary["folds"]]))))\n    (xtr, xte), scaler = scale_from_training(ds.x[tr], ds.x[te], enabled=scale_input)\n    refit_seed = stable_seed(seed, ds.task, outer_subject, "refit")\n    model = backend.fit_final(xtr, ds.y[tr], parameters, refit_seed, epochs,\n                              directory / "final_history.csv")\n    # This is the first point where the fitted outer model sees test inputs.\n    prediction = backend.predict(model, xte)\n    metrics = cycle_metrics(ds.y[te], prediction)\n    metadata = ds.cycles.iloc[te].reset_index(drop=True)\n    per_cycle = pd.concat([metadata, metrics], axis=1)\n    per_cycle.insert(0, "task", ds.task)\n    per_cycle.insert(1, "architecture", architecture)\n    per_cycle.insert(2, "seed", seed)\n    per_cycle.to_csv(directory / "cycle_metrics.csv", index=False)\n    np.savez_compressed(directory / "predictions.npz", measured=ds.y[te], predicted=prediction,\n                        cycle_ids=metadata.cycle_id.to_numpy(dtype=str),\n                        subject_ids=ds.groups[te], gait_phase=np.linspace(0, 100, ds.y.shape[1]))\n    backend.save(model, directory / "model.keras")\n    backend.clear()\n    summary = {"task": ds.task, "architecture": architecture, "seed": seed,\n               "subject": outer_subject, "n_cycles": len(te), "n_train_cycles": len(tr),\n               "n_train_subjects": len(set(ds.groups[tr])), "epochs": epochs,\n               "refit_seed": refit_seed, "rmse": float(metrics.rmse.mean()),\n               "nrmse": float(metrics.nrmse.mean()) if metrics.nrmse.notna().any() else None,\n               "mae": float(metrics.mae.mean()),\n               "undefined_nrmse_cycles": int((~metrics.nrmse_defined).sum()),\n               "inner_subject_mae": inner_summary["inner_subject_mae"],\n               "parameters": parameters, "input_scaler": scaler,\n               "epoch_rule": "ceiling of median best epoch across winning inner LOSO fits"}\n    # Written last: a completed outer fold can be safely skipped on resume.\n    write_json(directory / "result.json", summary)\n    return summary\n\n\ndef summarize_subject_results(records, directory, bootstrap_seed=2026, resamples=2000):\n    """Bootstrap participants; repeated seeds are averaged within participant first."""\n    directory = Path(directory)\n    df = pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, (list, dict))}\n                       for r in records])\n    if df.empty:\n        return\n    df.to_csv(directory / "outer_subject_metrics_by_seed.csv", index=False)\n    averaged = df.groupby(["task", "architecture", "subject"], as_index=False).agg(\n        rmse=("rmse", "mean"), nrmse=("nrmse", "mean"), mae=("mae", "mean"),\n        n_seeds=("seed", "nunique"), rmse_seed_sd=("rmse", "std"),\n        nrmse_seed_sd=("nrmse", "std"), mae_seed_sd=("mae", "std"))\n    averaged.to_csv(directory / "outer_subject_metrics_seed_average.csv", index=False)\n    rows = []\n    for (task, arch), group in averaged.groupby(["task", "architecture"]):\n        for metric in ["rmse", "nrmse", "mae"]:\n            values = group[metric].dropna().to_numpy()\n            if not len(values):\n                continue\n            rng = np.random.default_rng(stable_seed(bootstrap_seed, task, arch, metric))\n            if len(values) >= 2:\n                boot = np.median(rng.choice(values, size=(resamples, len(values)), replace=True), axis=1)\n                low, high = np.quantile(boot, [0.025, 0.975])\n            else:\n                low = high = np.nan\n            rows.append({"task": task, "architecture": arch, "metric": metric,\n                         "n_participants": len(values), "min_seeds_per_subject": int(group.n_seeds.min()),\n                         "max_seeds_per_subject": int(group.n_seeds.max()),\n                         "median": float(np.median(values)), "q25": float(np.quantile(values, .25)),\n                         "q75": float(np.quantile(values, .75)),\n                         "median_ci95_lower": low, "median_ci95_upper": high,\n                         "ci_method": "participant percentile bootstrap after averaging seeds"})\n    pd.DataFrame(rows).to_csv(directory / "cohort_summary.csv", index=False)\n', 'models.py': '"""TensorFlow model definitions and Bayesian selection over complete inner LOSO.\n\nTensorFlow/KerasTuner are imported only during training. Audit and data tests\ntherefore work in a lightweight environment without a GPU or TensorFlow.\n"""\nfrom __future__ import annotations\n\nimport gc\nfrom pathlib import Path\nimport numpy as np\n\nfrom protocol import evaluate_candidate, read_json, stable_seed, write_json\n\n\ndef training_dependencies():\n    try:\n        import tensorflow as tf\n        import keras_tuner as kt\n    except ImportError as exc:\n        raise RuntimeError("Training requires TensorFlow and keras-tuner. Install requirements.txt "\n                           "in a Python 3.11 environment; audit mode needs only requirements-audit.txt.") from exc\n    return tf, kt\n\n\ndef configure_tensorflow(seed, threads=0):\n    tf, _ = training_dependencies()\n    if threads:\n        tf.config.threading.set_inter_op_parallelism_threads(threads)\n        tf.config.threading.set_intra_op_parallelism_threads(threads)\n    for device in tf.config.list_physical_devices("GPU"):\n        tf.config.experimental.set_memory_growth(device, True)\n    tf.keras.utils.set_random_seed(seed)\n    tf.config.experimental.enable_op_determinism()\n    return {"tensorflow": tf.__version__, "devices": [d.name for d in tf.config.list_physical_devices()]}\n\n\ndef search_space(architecture, profile):\n    _, kt = training_dependencies()\n    hp = kt.HyperParameters()\n    if profile == "smoke":\n        hp.Fixed("num_layers", 1)\n        hp.Fixed("activation", "tanh")\n        hp.Fixed("optimizer", "adam")\n        hp.Fixed("learning_rate", 0.001)\n        hp.Fixed("loss", "mae")\n        hp.Fixed("encoder_units", 4)\n        hp.Fixed("decoder_units", 4)\n        hp.Fixed("units_0", 4)\n        return hp\n    max_layers = 10 if profile == "original" else 3\n    hp.Int("num_layers", 1, max_layers)\n    # The original GRU code registers tanh first under the shared HP name.\n    hp.Choice("activation", ["relu", "tanh"] if architecture == "conv1d_lstm" and profile == "original" else ["tanh"])\n    hp.Choice("optimizer", ["adam", "sgd", "rmsprop"] if profile == "original" else ["adam"])\n    hp.Choice("learning_rate", [0.0001, 0.001, 0.01] if profile == "original" else [0.0001, 0.001])\n    hp.Choice("loss", ["mse", "mae", "huber"] if architecture == "gru" else ["mse", "mae"])\n    hp.Fixed("encoder_units", (32 if architecture == "gru" else 100) if profile == "original" else 32)\n    if architecture != "gru":\n        hp.Choice("decoder_units", list(range(32, 513, 32)) if profile == "original" else [32, 64, 128])\n    unit_options = (list(range(32, 513, 32)) if architecture == "gru" else list(range(100, 513, 32)))\n    if profile != "original":\n        unit_options = [32, 64, 128]\n    for layer in range(max_layers):\n        with hp.conditional_scope("num_layers", list(range(layer + 1, max_layers + 1))):\n            hp.Choice(f"units_{layer}", unit_options)\n    return hp\n\n\ndef build_model(architecture, timesteps, parameters, profile="original"):\n    tf, _ = training_dependencies()\n    layers = tf.keras.layers\n    model = tf.keras.Sequential(name=architecture)\n    model.add(layers.Input(shape=(timesteps, 1)))\n    activation = parameters["activation"]\n    if architecture == "conv1d_lstm":\n        model.add(layers.Conv1D(4 if profile == "smoke" else 64, 3, activation="relu"))\n        model.add(layers.MaxPooling1D(pool_size=2))\n    if architecture in {"lstm", "conv1d_lstm"}:\n        model.add(layers.LSTM(parameters["encoder_units"], activation=activation))\n        model.add(layers.RepeatVector(timesteps))\n        model.add(layers.LSTM(parameters["decoder_units"], activation=activation, return_sequences=True))\n        for layer in range(parameters["num_layers"]):\n            model.add(layers.LSTM(parameters[f"units_{layer}"], activation=activation, return_sequences=True))\n        model.add(layers.TimeDistributed(layers.Dense(1)))\n    elif architecture == "gru":\n        model.add(layers.GRU(parameters["encoder_units"], activation=activation, return_sequences=True))\n        for layer in range(parameters["num_layers"]):\n            model.add(layers.GRU(parameters[f"units_{layer}"], activation=activation, return_sequences=True))\n        model.add(layers.Dense(1))\n    else:\n        raise ValueError(f"Unknown architecture: {architecture}")\n    optimizer = {"adam": tf.keras.optimizers.Adam, "sgd": tf.keras.optimizers.SGD,\n                 "rmsprop": tf.keras.optimizers.RMSprop}[parameters["optimizer"]](\n                     learning_rate=parameters["learning_rate"])\n    loss = tf.keras.losses.Huber() if parameters["loss"] == "huber" else parameters["loss"]\n    model.compile(optimizer=optimizer, loss=loss, metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")])\n    return model\n\n\nclass TensorFlowBackend:\n    def __init__(self, architecture, timesteps, profile, epochs, patience, batch_size, verbose):\n        self.tf, _ = training_dependencies()\n        self.architecture, self.timesteps, self.profile = architecture, timesteps, profile\n        self.epochs, self.patience = epochs, patience\n        self.batch_size, self.verbose = batch_size, verbose\n\n    def clear(self):\n        self.tf.keras.backend.clear_session()\n        gc.collect()\n\n    def fresh(self, parameters, seed):\n        self.clear()\n        self.tf.keras.utils.set_random_seed(seed)\n        return build_model(self.architecture, self.timesteps, parameters, self.profile)\n\n    def fit_inner(self, x_train, y_train, x_validation, y_validation, parameters, seed, history_path):\n        history_path = Path(history_path)\n        history_path.parent.mkdir(parents=True, exist_ok=True)\n        model = self.fresh(parameters, seed)\n        tf = self.tf\n        stop = tf.keras.callbacks.EarlyStopping(monitor="val_mae", mode="min",\n                                               patience=self.patience, min_delta=0,\n                                               restore_best_weights=True)\n        try:\n            history = model.fit(x_train, y_train,\n                                validation_data=(x_validation, y_validation),\n                                epochs=self.epochs, batch_size=self.batch_size, shuffle=True,\n                                callbacks=[stop, tf.keras.callbacks.TerminateOnNaN(),\n                                           tf.keras.callbacks.CSVLogger(str(history_path))],\n                                verbose=self.verbose)\n            values = np.asarray(history.history["val_mae"], dtype=float)\n            if not len(values) or not np.isfinite(values).all():\n                raise ValueError("Nonfinite inner training/validation history; trial must fail")\n            best_epoch = int(np.argmin(values)) + 1\n            prediction = self.predict(model, x_validation)\n            return {"prediction": prediction, "best_epoch": best_epoch, "epochs_run": len(values)}\n        finally:\n            del model\n            self.clear()\n\n    def fit_final(self, x_train, y_train, parameters, seed, epochs, history_path):\n        """No validation split or validation-driven callback in the outer refit."""\n        model = self.fresh(parameters, seed)\n        tf = self.tf\n        history = model.fit(x_train, y_train, epochs=epochs, batch_size=self.batch_size,\n                            shuffle=True, callbacks=[tf.keras.callbacks.TerminateOnNaN(),\n                                                      tf.keras.callbacks.CSVLogger(str(history_path))],\n                            verbose=self.verbose)\n        if len(history.history["loss"]) != epochs or not np.isfinite(history.history["loss"]).all():\n            raise ValueError("Outer refit did not complete with finite losses")\n        return model\n\n    def predict(self, model, inputs):\n        return np.asarray(model.predict(inputs, batch_size=self.batch_size, verbose=0))\n\n    def save(self, model, path):\n        model.save(path)\n        lines = []\n        model.summary(print_fn=lambda line, **kwargs: lines.append(line))\n        Path(path).with_suffix(".summary.txt").write_text("\\n".join(lines), encoding="utf-8")\n\n\ndef select_hyperparameters(ds, outer_subject, architecture, backend, seed, directory,\n                           profile, max_trials, scale_input, resume=False):\n    """A Bayesian trial is a COMPLETE inner LOSO, never just one validation subject."""\n    _, kt = training_dependencies()\n    directory = Path(directory)\n    trial_root = directory / "candidate_fits"\n    trial_root.mkdir(parents=True, exist_ok=True)\n\n    class InnerLOSOTuner(kt.Tuner):\n        def run_trial(self, trial, *args, **kwargs):\n            parameters = dict(trial.hyperparameters.values)\n            trial_dir = trial_root / trial.trial_id\n            print(f"  {ds.task}/{architecture}/{outer_subject}: trial {trial.trial_id}", flush=True)\n            try:\n                summary = evaluate_candidate(ds, outer_subject, parameters, backend, seed,\n                                             trial_dir, scale_input)\n            except (ValueError, backend.tf.errors.ResourceExhaustedError) as exc:\n                write_json(trial_dir / "failure.json", {"error": str(exc), "parameters": parameters})\n                # Do not average only the successful validation participants.\n                raise kt.errors.FailedTrialError(str(exc)) from exc\n            return {"inner_subject_mae": summary["inner_subject_mae"]}\n\n    hp = search_space(architecture, profile)\n    oracle = kt.oracles.BayesianOptimizationOracle(\n        objective=kt.Objective("inner_subject_mae", direction="min"),\n        max_trials=max_trials, num_initial_points=min(3, max_trials),\n        hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,\n        seed=stable_seed(seed, ds.task, architecture, outer_subject, "oracle"),\n        max_retries_per_trial=0, max_consecutive_failed_trials=3)\n    tuner = InnerLOSOTuner(oracle=oracle, hypermodel=None, directory=str(directory),\n                          project_name="tuner", overwrite=not resume)\n    tuner.search()\n    best = tuner.oracle.get_best_trials(num_trials=1)\n    if not best or best[0].score is None or not np.isfinite(best[0].score):\n        raise RuntimeError("No successful full inner LOSO trial; inspect failure.json files")\n    trial = best[0]\n    summary = read_json(trial_root / trial.trial_id / "inner_summary.json")\n    write_json(directory / "selection.json", {\n        "trial_id": trial.trial_id, "inner_subject_mae": summary["inner_subject_mae"],\n        "parameters": summary["parameters"], "search_profile": profile,\n        "n_inner_folds": len(summary["folds"])})\n    return summary["parameters"], summary\n', 'revision_data.py': '"""Strict cohort contract and coordinate controls; no guessed anatomical data."""\nfrom dataclasses import dataclass, replace\nfrom hashlib import sha256\nfrom pathlib import Path\nimport json\nimport numpy as np\nimport pandas as pd\nfrom data import Dataset, load_datasets\nfrom protocol import write_json, read_json\n\nMODES = (\'absolute\', \'translation\', \'pelvis\', \'aligned\', \'body\', \'combined\')\nAXIS = {\'AP\': 0, \'ML\': 1, \'Vertical\': 2}\nREQUIRED_CONTEXT = [\'cohort_id\', \'dataset_role\', \'ankle_units\', \'grf_units\',\n                    \'coordinate_definition\', \'motion_capture_system\', \'force_platforms\',\n                    \'kinematics_sampling_hz\', \'force_sampling_hz\', \'walking_protocol\',\n                    \'harmonization_description\', \'cycle_definition\']\n\ndef digest_file(path):\n    h = sha256()\n    with Path(path).open(\'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef bool_column(series):\n    mapping = {\'true\': True, \'1\': True, \'yes\': True, \'false\': False, \'0\': False, \'no\': False}\n    v = series.astype(str).str.lower().str.strip().map(mapping)\n    if v.isna().any():\n        raise ValueError(\'include must contain explicit true/false values\')\n    return v.astype(bool)\n\ndef read_table(path, columns):\n    df = pd.read_csv(path, dtype=str, keep_default_na=False)\n    if set(columns) - set(df):\n        raise ValueError(f\'{path}: missing columns {set(columns) - set(df)}\')\n    for col in set(columns) - {\'exclusion_reason\'}:\n        if df[col].str.strip().eq(\'\').any():\n            raise ValueError(f\'{path}: blank required {col}\')\n    return df\n\n@dataclass\nclass Cohort:\n    ds: Dataset\n    context: dict\n    participants: pd.DataFrame\n    trials: pd.DataFrame\n    ledger: pd.DataFrame\n    arrays: dict\n\ndef legacy_cohort(archive, task):\n    ds = load_datasets(archive, [task])[task]\n    meta = ds.cycles.copy()\n    meta[\'participant_id\'] = meta.subject\n    meta[\'trial_id\'] = \'unknown\'  # Never invent trial IDs from cycle positions.\n    meta[\'visit\'] = \'internal\'\n    meta[\'side\'] = task.split(\'_\')[-1].lower()\n    meta[\'include\'] = True\n    meta[\'exclusion_reason\'] = \'\'\n    ds = replace(ds, cycles=meta)\n    context = {k: \'unknown\' for k in REQUIRED_CONTEXT}\n    context.update(cohort_id=\'Shida_internal\', dataset_role=\'internal\', ankle_units=\'mm\',\n                   grf_units=\'N/kg\', units_verified=False,\n                   coordinate_definition=\'Original single component; physical frame unverified\',\n                   source=\'Paired Raw sheets; manuscript units assumed, not independently verified\')\n    people = pd.DataFrame({\'participant_id\': ds.subjects})\n    return Cohort(ds, context, people, pd.DataFrame(), meta, {})\n\ndef load_cohort(directory, task, visits=None, role=None):\n    """Read one side/component cohort bundle. Excluded cycles need no waveform."""\n    root = Path(directory)\n    context = read_json(root / \'cohort.json\')\n    if any(k not in context or context[k] is None for k in REQUIRED_CONTEXT):\n        raise ValueError(\'Complete cohort.json; use "unknown" explicitly for unavailable descriptive fields\')\n    if role and context[\'dataset_role\'] != role:\n        raise ValueError(\'Wrong dataset_role\')\n    if context[\'ankle_units\'] not in (\'m\', \'mm\') or context[\'grf_units\'] != \'N/kg\':\n        raise ValueError(\'Convert ankle units to m/mm and GRF to N/kg before making the bundle\')\n    if context.get(\'task\') != task:\n        raise ValueError(\'cohort.json task differs from requested component/side\')\n    people = read_table(root / \'participants.csv\', [\'participant_id\'])\n    trials = read_table(root / \'trials.csv\', [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    ledger = read_table(root / \'cycles.csv\', [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'])\n    if people.participant_id.duplicated().any() or ledger.cycle_id.duplicated().any():\n        raise ValueError(\'Participant and cycle IDs must be unique within this cohort bundle\')\n    if not people.participant_id.str.fullmatch(r\'[A-Za-z0-9_-]+\').all():\n        raise ValueError(\'Use letters, digits, underscores, or hyphens in participant IDs\')\n    keys = [\'participant_id\', \'trial_id\', \'visit\', \'side\']\n    if trials.duplicated(keys).any():\n        raise ValueError(\'Duplicate trial record\')\n    for frame in (trials, ledger):\n        frame[\'include\'] = bool_column(frame.include)\n        if frame.loc[~frame.include, \'exclusion_reason\'].str.strip().eq(\'\').any():\n            raise ValueError(\'Every excluded trial/cycle requires a reason\')\n        if set(frame.participant_id) - set(people.participant_id):\n            raise ValueError(\'Participant missing from participants.csv\')\n        if not set(frame.visit).issubset({\'baseline\', \'post_intervention\', \'internal\'}):\n            raise ValueError(\'Use baseline, post_intervention, or internal visit labels\')\n        if set(frame.side) != {task.split(\'_\')[-1].lower()}:\n            raise ValueError(\'Each bundle must describe just its selected side\')\n    joined = ledger.merge(trials[keys + [\'include\']], on=keys, how=\'left\', suffixes=(\'\', \'_trial\'), validate=\'many_to_one\')\n    if joined.include_trial.isna().any() or (joined.include & ~joined.include_trial.astype(bool)).any():\n        raise ValueError(\'Every cycle must link to a trial; an excluded trial cannot have an included cycle\')\n    with np.load(root / \'waveforms.npz\', allow_pickle=False) as f:\n        arrays = {k: f[k] for k in f.files}\n    ids = arrays.pop(\'cycle_ids\').astype(str)\n    valid = ledger[ledger.include].copy().set_index(\'cycle_id\', drop=False)\n    if len(set(ids)) != len(ids) or set(ids) != set(valid.index):\n        raise ValueError(\'NPZ cycle_ids must match ALL included cycles, one-to-one, before visit filtering\')\n    meta = valid.loc[ids].reset_index(drop=True)\n    n = len(ids)\n    for k, arr in arrays.items():\n        if len(arr) != n or not np.issubdtype(arr.dtype, np.number) or not np.isfinite(arr).all():\n            raise ValueError(f\'{k}: wrong cycle count or nonfinite/nonnumeric values\')\n    if \'ankle_xyz\' in arrays:\n        xyz = arrays[\'ankle_xyz\']\n        if xyz.shape != (n, 100, 3) or arrays.get(\'grf_xyz\', np.empty(0)).shape != xyz.shape:\n            raise ValueError(\'ankle_xyz and grf_xyz must both be [N,100,3]\')\n        if context.get(\'canonical_axis_order\') != [\'forward\', \'left\', \'up\']:\n            raise ValueError(\'Map both kinematics and GRFs into the same right-handed [forward,left,up] laboratory axes\')\n        if arrays.get(\'direction_xyz\', np.empty(0)).shape != (n, 3):\n            raise ValueError(\'Provide a force-independent horizontal walking direction for every cycle\')\n        basis = gait_basis(arrays[\'direction_xyz\'])\n        axis = AXIS[task.split(\'_\')[0]]\n        x = xyz[:, :, axis:axis + 1]\n        # SAME gait-frame targets for ALL coordinate variants, enabling fair comparisons.\n        y = np.einsum(\'nti,nij->ntj\', arrays[\'grf_xyz\'], basis)[:, :, axis:axis + 1]\n    else:\n        x, y = arrays.get(\'x\'), arrays.get(\'y\')\n        if x is None or y is None or x.shape != (n, 100, 1) or y.shape != x.shape:\n            raise ValueError(\'Scalar bundles require x and y, each [N,100,1]\')\n    take = np.ones(n, bool) if visits is None else meta.visit.isin(visits).to_numpy()\n    if not take.any():\n        raise ValueError(\'No valid cycles for the prespecified visit policy\')\n    meta = meta.loc[take].reset_index(drop=True)\n    meta[\'subject\'] = meta.participant_id\n    fingerprint = sha256(\'\'.join(digest_file(root / f) for f in\n                         [\'cohort.json\', \'participants.csv\', \'trials.csv\', \'cycles.csv\', \'waveforms.npz\']).encode()).hexdigest()\n    fingerprint = sha256((fingerprint + repr(visits)).encode()).hexdigest()\n    ds = Dataset(task, x[take].astype(\'float32\'), y[take].astype(\'float32\'),\n                 meta.participant_id.to_numpy(str), meta, fingerprint)\n    return Cohort(ds, context, people, trials, ledger, {k: v[take] for k, v in arrays.items()})\n\ndef gait_basis(direction):\n    forward = np.array(direction, float, copy=True)\n    if forward.ndim != 2 or forward.shape[1] != 3 or not np.isfinite(forward).all():\n        raise ValueError(\'Invalid direction vectors\')\n    if np.any(np.abs(forward[:, 2]) > 1e-6):\n        raise ValueError(\'Walking directions must be projected onto the horizontal plane\')\n    length = np.linalg.norm(forward, axis=1)\n    if np.any(length < 1e-8):\n        raise ValueError(\'Zero walking direction; supply verified direction from pelvis progression/protocol\')\n    forward /= length[:, None]\n    up = np.tile([0., 0., 1.], (len(forward), 1))\n    left = np.cross(up, forward)\n    return np.stack([forward, left, up], axis=2)\n\ndef coordinate_dataset(cohort, mode=\'absolute\', translation_m=None, rotation_deg=0):\n    if mode not in MODES:\n        raise ValueError(f\'Unknown coordinate mode {mode}\')\n    ds, a = cohort.ds, cohort.arrays\n    units = cohort.context[\'ankle_units\']\n    factor = 1000. if units == \'mm\' else 1.\n    axis = AXIS[ds.task.split(\'_\')[0]]\n    if \'ankle_xyz\' not in a:\n        if mode not in (\'absolute\', \'translation\') or rotation_deg:\n            raise ValueError(\'This analysis requires verified 3D/pelvis/body-scale data; scalar inputs cannot supply it\')\n        x = ds.x.astype(float).copy()\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            x += np.asarray(translation_m)[axis] * factor\n        if mode == \'translation\':\n            x -= x[:, :1, :]\n    else:\n        x = a[\'ankle_xyz\'].astype(float).copy()\n        pelvis = a.get(\'pelvis_xyz\')\n        pelvis = None if pelvis is None else pelvis.astype(float).copy()\n        direction = a[\'direction_xyz\'].astype(float).copy()\n        if rotation_deg:\n            theta = np.deg2rad(rotation_deg)\n            R = np.array([[np.cos(theta), -np.sin(theta), 0], [np.sin(theta), np.cos(theta), 0], [0, 0, 1]])\n            x = x @ R.T\n            direction = direction @ R.T\n            if pelvis is not None:\n                pelvis = pelvis @ R.T\n        if translation_m is not None:\n            if not cohort.context.get(\'units_verified\', False):\n                raise ValueError(\'Verify physical units before metre-based perturbations\')\n            offset = np.asarray(translation_m, float) * factor\n            if offset.shape != (3,):\n                raise ValueError(\'Translation requires three metre offsets\')\n            x += offset\n            if pelvis is not None:\n                pelvis += offset\n        if mode in (\'pelvis\', \'combined\'):\n            if pelvis is None or pelvis.shape != x.shape:\n                raise ValueError(\'Synchronized pelvis_xyz [N,100,3] is required\')\n            x -= pelvis\n        elif mode in (\'translation\', \'aligned\', \'body\'):\n            x -= x[:, :1, :]\n        if mode in (\'aligned\', \'combined\'):\n            x = np.einsum(\'nti,nij->ntj\', x, gait_basis(direction))\n        if mode in (\'body\', \'combined\'):\n            if not cohort.context.get(\'units_verified\', False) or not cohort.context.get(\'body_scale_definition\'):\n                raise ValueError(\'Verify coordinate units and prespecify height or leg-length definition\')\n            height = a.get(\'body_scale_m\')\n            if height is None or height.shape != (len(x),) or np.any(height <= 0):\n                raise ValueError(\'Positive body_scale_m [N], with a fixed definition, is required\')\n            x /= height[:, None, None] * factor\n        x = x[:, :, axis:axis + 1]\n    fingerprint = sha256((ds.fingerprint + repr((mode, translation_m, rotation_deg))).encode()).hexdigest()\n    return replace(ds, x=x.astype(\'float32\'), fingerprint=fingerprint)\n\ndef write_templates(directory):\n    """Blank data contract, not fabricated external observations."""\n    root = Path(directory)\n    root.mkdir(parents=True, exist_ok=True)\n    for name, columns in {\n        \'participants.csv\': [\'participant_id\', \'global_subject_id\', \'age\', \'sex\', \'medication_state\', \'HY_stage\', \'intervention_group\'],\n        \'trials.csv\': [\'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n        \'cycles.csv\': [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\', \'include\', \'exclusion_reason\'],\n    }.items():\n        if not (root / name).exists():\n            pd.DataFrame(columns=columns).to_csv(root / name, index=False)\n    context = {k: None for k in REQUIRED_CONTEXT}\n    context.update(task=None, units_verified=False, harmonization_verified=False, participant_disjointness_verified=False,\n                   canonical_axis_order=[\'forward\', \'left\', \'up\'], body_scale_definition=None)\n    if not (root / \'cohort.json\').exists():\n        write_json(root / \'cohort.json\', context)\n    return root\n\ndef harmonize_input_units(cohort, training_units):\n    """Deterministic physical-unit conversion, never fitted on external data."""\n    units = cohort.context[\'ankle_units\']\n    if units not in (\'m\', \'mm\') or training_units not in (\'m\', \'mm\'):\n        raise ValueError(\'Known internal and external length units are required\')\n    if units == training_units:\n        return cohort\n    multiplier = 1000. if units == \'m\' else .001\n    arrays = dict(cohort.arrays)\n    for key in [\'ankle_xyz\', \'pelvis_xyz\']:\n        if key in arrays:\n            arrays[key] = arrays[key] * multiplier\n    ds = replace(cohort.ds, x=cohort.ds.x * multiplier)\n    return replace(cohort, ds=ds, arrays=arrays,\n                   context={**cohort.context, \'ankle_units\': training_units})\n\ndef pack_scalar_workbooks(input_xlsx, target_xlsx, cycle_map_csv, participants_csv,\n                          trials_csv, cohort_json, output):\n    """Convert explicitly mapped old-format Excel cycles; never infer participants."""\n    import shutil\n    root = Path(output)\n    root.mkdir(parents=True, exist_ok=True)\n    ledger = read_table(cycle_map_csv, [\'cycle_id\', \'participant_id\', \'trial_id\', \'visit\', \'side\',\n                                       \'include\', \'exclusion_reason\'])\n    if {\'input_column\', \'target_column\'} - set(ledger):\n        raise ValueError(\'Explicit input_column and target_column mappings are required\')\n    accepted = ledger[bool_column(ledger.include)]\n    inputs = pd.read_excel(input_xlsx, engine=\'openpyxl\')\n    targets = pd.read_excel(target_xlsx, engine=\'openpyxl\')\n    if inputs.shape[0] != 100 or targets.shape[0] != 100:\n        raise ValueError(\'Prepare matched complete cycles at 100 normalized gait phases before packing; no padding/resampling is guessed\')\n    xs, ys = [], []\n    for row in accepted.itertuples():\n        x = inputs[row.input_column].to_numpy(dtype=\'float32\')\n        y = targets[row.target_column].to_numpy(dtype=\'float32\')\n        if not np.isfinite(x).all() or not np.isfinite(y).all():\n            raise ValueError(f\'{row.cycle_id}: missing values; document exclusion or fix the source, never zero-fill\')\n        xs.append(x[:, None]); ys.append(y[:, None])\n    if not xs:\n        raise ValueError(\'No included cycles\')\n    np.savez_compressed(root / \'waveforms.npz\', cycle_ids=accepted.cycle_id.to_numpy(str), x=np.stack(xs), y=np.stack(ys))\n    ledger.to_csv(root / \'cycles.csv\', index=False)\n    for source, filename in [(participants_csv, \'participants.csv\'), (trials_csv, \'trials.csv\'), (cohort_json, \'cohort.json\')]:\n        if Path(source).resolve() != (root / filename).resolve():\n            shutil.copy2(source, root / filename)\n    context = read_json(root / \'cohort.json\')\n    load_cohort(root, context[\'task\'], role=context[\'dataset_role\'])\n    return root\n', 'revision_models.py': '"""Subject-wise model-family selection and four explicit comparator families."""\nfrom pathlib import Path\nimport json\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.linear_model import Ridge\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.exceptions import ConvergenceWarning\nfrom protocol import cycle_metrics, stable_seed, scale_from_training, write_json, read_json, parameter_hash\n\nBASELINES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\')\nDEEP = (\'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef participant_mean(y, groups):\n    return np.mean([y[groups == g].mean(axis=0) for g in np.unique(groups)], axis=0)\n\ndef fourier_basis(timesteps, harmonics):\n    # Endpoints include 0 and 100% consistently with the supplied waveforms.\n    phase = np.linspace(0, 1, timesteps)\n    return np.column_stack([np.ones(timesteps)] +\n        [f(2 * np.pi * k * phase) for k in range(1, harmonics + 1) for f in (np.sin, np.cos)])\n\ndef baseline_grid(family, smoke=False):\n    if family == \'mean_waveform\':\n        return [{}]\n    if family == \'ridge\':\n        return [{\'alpha\': v} for v in ([1.] if smoke else [0.01, 0.1, 1., 10., 100.])]\n    if family == \'phase_only\':\n        return [{\'harmonics\': v} for v in ([3] if smoke else [1, 3, 5, 10])]\n    if family == \'shallow_mlp\':\n        return [{\'units\': u, \'alpha\': a, \'max_iter\': 30 if smoke else 1000}\n                for u in ([4] if smoke else [16, 32, 64])\n                for a in ([0.01] if smoke else [0.0001, 0.01])]\n    raise ValueError(family)\n\ndef fit_baseline(family, x, y, groups, parameters, seed):\n    obj = {\'family\': family, \'shape\': list(y.shape[1:]), \'parameters\': parameters}\n    if family == \'mean_waveform\':\n        obj[\'waveform\'] = participant_mean(y, groups)\n    elif family == \'phase_only\':\n        basis = fourier_basis(y.shape[1], parameters[\'harmonics\'])\n        # Equal participant weighting. Uses phase alone, never ankle positions.\n        coef = np.linalg.lstsq(basis, participant_mean(y, groups), rcond=None)[0]\n        obj[\'waveform\'] = basis @ coef\n    else:\n        flat = x.reshape(len(x), -1)\n        scaler = StandardScaler().fit(flat)  # Training rows only.\n        z = scaler.transform(flat)\n        target = y.reshape(len(y), -1)\n        if family == \'ridge\':\n            model = Ridge(alpha=parameters[\'alpha\'])\n            weights = np.array([1. / np.sum(groups == g) for g in groups])\n            weights *= len(weights) / weights.sum()\n            model.fit(z, target, sample_weight=weights)\n            obj[\'fit_weighting\'] = \'equal participants\'\n        elif family == \'shallow_mlp\':\n            # One hidden layer; training-loss optimizer convergence, NO random validation split.\n            model = MLPRegressor(hidden_layer_sizes=(parameters[\'units\'],), activation=\'relu\',\n                                 solver=\'lbfgs\', alpha=parameters[\'alpha\'], max_iter=parameters[\'max_iter\'],\n                                 max_fun=50000, early_stopping=False, random_state=seed)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter(\'always\', ConvergenceWarning)\n                model.fit(z, target)\n            obj[\'convergence_warnings\'] = [str(w.message) for w in caught]\n            obj[\'fit_weighting\'] = \'equal cycles; validation score remains equal participants\'\n        else:\n            raise ValueError(family)\n        obj.update(model=model, scaler=scaler)\n    return obj\n\ndef predict_baseline(obj, x):\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x), -1))\n    return obj[\'model\'].predict(z).reshape((len(x), *obj[\'shape\']))\n\ndef backend(family, ds, cfg):\n    from models import TensorFlowBackend\n    return TensorFlowBackend(family, ds.x.shape[1], cfg[\'search_profile\'], cfg[\'epochs\'],\n                             cfg[\'patience\'], cfg[\'batch_size\'], cfg[\'verbose\'])\n\ndef cv_candidate(ds, family, parameters, cfg, directory):\n    """Only the development Dataset is passed; outer/external arrays are absent."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    signature = parameter_hash(parameters)\n    rows = []\n    for val in ds.subjects:\n        path = directory / f\'validation_{val}.json\'\n        if path.exists():\n            row = read_json(path)\n            if row[\'parameter_hash\'] != signature or row[\'data_fingerprint\'] != ds.fingerprint:\n                raise ValueError(\'Candidate cache changed\')\n            rows.append(row)\n            continue\n        tr, va = ds.groups != val, ds.groups == val\n        if not tr.any():\n            raise ValueError(\'At least two development participants are required\')\n        seed = stable_seed(cfg[\'seed\'], ds.task, family, val, \'inner\')\n        print(f\'    {family}: validation {val}\', flush=True)\n        if family in DEEP:\n            b = backend(family, ds, cfg)\n            (xt, xv), _ = scale_from_training(ds.x[tr], ds.x[va], enabled=cfg[\'scale_input\'])\n            fit = b.fit_inner(xt, ds.y[tr], xv, ds.y[va], parameters, seed, directory / f\'history_{val}.csv\')\n            prediction, epoch = fit[\'prediction\'], fit[\'best_epoch\']\n            convergence = []\n        else:\n            model = fit_baseline(family, ds.x[tr], ds.y[tr], ds.groups[tr], parameters, seed)\n            prediction, epoch = predict_baseline(model, ds.x[va]), None\n            convergence = model.get(\'convergence_warnings\', [])\n        metric = cycle_metrics(ds.y[va], np.asarray(prediction))\n        row = {\'validation_subject\': val, \'training_subjects\': sorted(set(ds.groups[tr])),\n               \'n_validation_cycles\': int(va.sum()), \'parameter_hash\': signature,\n               \'data_fingerprint\': ds.fingerprint, \'mae\': float(metric.mae.mean()),\n               \'rmse\': float(metric.rmse.mean()), \'best_epoch\': epoch, \'convergence_warnings\': convergence}\n        write_json(path, row)\n        rows.append(row)\n    score = float(np.mean([r[\'mae\'] for r in rows]))\n    if not np.isfinite(score):\n        raise ValueError(\'Nonfinite full subject-wise validation objective\')\n    result = {\'family\': family, \'parameters\': parameters, \'inner_subject_mae\': score, \'folds\': rows,\n              \'epochs\': max(1, int(np.ceil(np.median([r[\'best_epoch\'] for r in rows])))) if family in DEEP else None}\n    write_json(directory / \'summary.json\', result)\n    return result\n\ndef tune_family(ds, family, cfg, directory):\n    directory = Path(directory)\n    done = directory / \'selected.json\'\n    if done.exists():\n        return read_json(done)\n    directory.mkdir(parents=True, exist_ok=True)\n    if family not in DEEP:\n        candidates = [cv_candidate(ds, family, p, cfg, directory / f\'candidate_{i:03d}\')\n                      for i, p in enumerate(baseline_grid(family, cfg.get(\'smoke\', False)))]\n        selected = min(candidates, key=lambda r: (r[\'inner_subject_mae\'], parameter_hash(r[\'parameters\'])))\n    else:\n        from models import training_dependencies, search_space\n        tf, kt = training_dependencies()\n        class SubjectTuner(kt.Tuner):\n            def run_trial(self, trial, *args, **kwargs):\n                try:\n                    result = cv_candidate(ds, family, dict(trial.hyperparameters.values), cfg,\n                                          directory / \'candidates\' / trial.trial_id)\n                except (ValueError, tf.errors.ResourceExhaustedError) as exc:\n                    write_json(directory / \'candidates\' / trial.trial_id / \'failure.json\', {\'error\': str(exc)})\n                    raise kt.errors.FailedTrialError(str(exc)) from exc\n                return {\'inner_subject_mae\': result[\'inner_subject_mae\']}\n        hp = search_space(family, cfg[\'search_profile\'])\n        oracle = kt.oracles.BayesianOptimizationOracle(objective=kt.Objective(\'inner_subject_mae\', \'min\'),\n            max_trials=cfg[\'max_trials\'], num_initial_points=min(3, cfg[\'max_trials\']),\n            hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,\n            seed=stable_seed(cfg[\'seed\'], ds.task, family, \'oracle\'), max_consecutive_failed_trials=3)\n        tuner = SubjectTuner(oracle=oracle, hypermodel=None, directory=str(directory), project_name=\'tuner\', overwrite=False)\n        tuner.search()\n        best = tuner.oracle.get_best_trials(1)\n        if not best or best[0].score is None or not np.isfinite(best[0].score):\n            raise ValueError(f\'No complete successful candidate for {family}\')\n        selected = read_json(directory / \'candidates\' / best[0].trial_id / \'summary.json\')\n    write_json(done, selected)\n    return selected\n\ndef fit_frozen_model(ds, selected, cfg, directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    descriptor = directory / \'model.json\'\n    if descriptor.exists():\n        return read_json(descriptor)\n    family = selected[\'family\']\n    seed = stable_seed(cfg[\'seed\'], ds.task, family, \'final_refit\')\n    if family in DEEP:\n        b = backend(family, ds, cfg)\n        (x,), scaler = scale_from_training(ds.x, enabled=cfg[\'scale_input\'])\n        model = b.fit_final(x, ds.y, selected[\'parameters\'], seed, selected[\'epochs\'], directory / \'history.csv\')\n        b.save(model, directory / \'model.keras\')\n        b.clear()\n        result = {\'family\': family, \'file\': \'model.keras\', \'input_scaler\': scaler}\n    else:\n        model = fit_baseline(family, ds.x, ds.y, ds.groups, selected[\'parameters\'], seed)\n        joblib.dump(model, directory / \'model.joblib\')\n        result = {\'family\': family, \'file\': \'model.joblib\',\n                  \'convergence_warnings\': model.get(\'convergence_warnings\', [])}\n    result.update(parameters=selected[\'parameters\'], epochs=selected[\'epochs\'], seed=seed,\n                  n_training_participants=len(ds.subjects), n_training_cycles=len(ds.x),\n                  training_participants=ds.subjects, data_fingerprint=ds.fingerprint)\n    write_json(descriptor, result)\n    return result\n\ndef predict_saved(directory, x):\n    directory = Path(directory)\n    info = read_json(directory / \'model.json\')\n    if info[\'family\'] in DEEP:\n        from models import training_dependencies\n        tf, _ = training_dependencies()\n        scaler = info[\'input_scaler\']\n        z = (x - np.array(scaler[\'mean\'])[None, None]) / np.array(scaler[\'std\'])[None, None]\n        model = tf.keras.models.load_model(directory / info[\'file\'], compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    # Only load models created by this notebook, whose file hashes are frozen.\n    return np.asarray(predict_baseline(joblib.load(directory / info[\'file\']), x))\n\ndef mean_ensemble(predictions):\n    """Average predictions BEFORE errors: models are not independent participants."""\n    values = [np.asarray(p) for p in predictions]\n    if not values or any(v.shape != values[0].shape or not np.isfinite(v).all() for v in values):\n        raise ValueError(\'Ensemble members require identical finite cycle/time/component arrays\')\n    return np.mean(np.stack(values), axis=0)\n', 'evaluate_saved.py': '"""Read-only held-out evaluation. This module never trains or tunes a model."""\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport numpy as np\nimport pandas as pd\nfrom protocol import cycle_metrics\n\nFAMILIES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\', \'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef read_json(path):\n    return json.loads(Path(path).read_text())\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024*1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\ndef check_metadata(spec, info, selected, family, outer, ds):\n    if spec[\'config\'].get(\'smoke\', False):\n        raise ValueError(\'This evaluator requires research outputs, not smoke-test fixtures\')\n    if spec[\'task\'] != ds.task or spec[\'data_fingerprint\'] != ds.fingerprint:\n        raise ValueError(\'Internal data or preprocessing differs from the saved experiment\')\n    if sorted(spec[\'subjects\']) != ds.subjects or outer not in ds.subjects:\n        raise ValueError(\'Participant list differs from the saved experiment\')\n    if info[\'family\'] != family or selected[\'family\'] != family:\n        raise ValueError(\'Saved family mismatch\')\n    training = info.get(\'training_participants\', [])\n    expected = [s for s in ds.subjects if s != outer]\n    if outer in training or sorted(training) != expected or len(training) != len(set(training)):\n        raise ValueError(\'Test participant overlaps training, or training cohort is incomplete\')\n    if info.get(\'n_training_participants\') != len(expected):\n        raise ValueError(\'Saved training-participant count differs\')\n    if info.get(\'data_fingerprint\') != ds.subset(expected).fingerprint:\n        raise ValueError(\'Model training-data fingerprint does not match\')\n    if info.get(\'parameters\') != selected.get(\'parameters\'):\n        raise ValueError(\'Saved refit parameters differ from the frozen selection\')\n    if family in (\'lstm\', \'gru\', \'conv1d_lstm\') and info.get(\'epochs\') != selected.get(\'epochs\'):\n        raise ValueError(\'Saved refit epochs differ from the frozen selection\')\n\ndef read_prediction_cache(path, te):\n    with np.load(path, allow_pickle=False) as cache:\n        ids = cache[\'cycle_ids\'].astype(str)\n        truth = cache[\'measured\']\n        pred = cache[\'predicted\']\n    expected = te.cycles.cycle_id.to_numpy(str)\n    if not np.array_equal(ids, expected):\n        raise ValueError(\'Saved predictions have different cycle identities/order\')\n    if truth.shape != te.y.shape or pred.shape != te.y.shape:\n        raise ValueError(\'Saved prediction or target shape differs from test data\')\n    if not np.isfinite(truth).all() or not np.isfinite(pred).all():\n        raise ValueError(\'Nonfinite cached targets or predictions\')\n    if not np.allclose(truth, te.y, rtol=1e-6, atol=1e-7):\n        raise ValueError(\'Cached targets differ from the original held-out measurements\')\n    return pred\n\ndef predict_weights(directory, info, x, software):\n    """Inference only, using the input scaler saved with the fitted model."""\n    path = Path(directory)/info[\'file\']\n    if info[\'family\'] in (\'lstm\',\'gru\',\'conv1d_lstm\'):\n        for name in [\'tensorflow\',\'keras\']:\n            actual = importlib.metadata.version(name)\n            if actual != software.get(name):\n                raise ValueError(f\'{name}: inference needs the saved version {software.get(name)}, found {actual}. Cached predictions do not require TensorFlow.\')\n        import tensorflow as tf\n        scaler = info[\'input_scaler\']\n        std = np.asarray(scaler[\'std\'], float)\n        if not np.isfinite(std).all() or np.any(std <= 0):\n            raise ValueError(\'Invalid saved scaler\')\n        z = (x - np.asarray(scaler[\'mean\'])[None,None,:]) / std[None,None,:]\n        model = tf.keras.models.load_model(path, compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    for name in [\'scikit-learn\',\'joblib\']:\n        actual = importlib.metadata.version(name)\n        if actual != software.get(name):\n            raise ValueError(f\'{name}: saved-model version is {software.get(name)}, current version is {actual}. Use cached predictions or the original package versions.\')\n    import joblib\n    # Only load trusted model files created by the user\'s own training notebook.\n    obj = joblib.load(path)\n    if obj[\'family\'] != info[\'family\']:\n        raise ValueError(\'Baseline family differs inside saved model\')\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x),-1))\n    return np.asarray(obj[\'model\'].predict(z)).reshape((len(x),*obj[\'shape\']))\n\ndef evaluate_available(root, ds, outer, output, mode=\'cached_only\'):\n    if mode not in (\'cached_only\',\'inference_if_needed\',\'fresh_inference\'):\n        raise ValueError(\'Unknown evaluation mode\')\n    root, output = Path(root), Path(output)\n    if output.resolve().is_relative_to(root.resolve()):\n        raise ValueError(\'Write test reports outside the training experiment\')\n    output.mkdir(parents=True,exist_ok=True)\n    spec = read_json(root/\'prespecification.json\')\n    te = ds.subset([outer])\n    inventory, tables, summary, predictions, hashes = [], [], [], {}, {}\n    for family in FAMILIES:\n        folder = root/\'nested\'/outer/family\n        descriptor, selection = folder/\'refit/model.json\', folder/\'selection/selected.json\'\n        if not descriptor.exists() or not selection.exists():\n            inventory.append(dict(model=family,status=\'unavailable\',reason=\'No completed frozen selection/refitted model\'))\n            continue\n        try:\n            info, chosen = read_json(descriptor), read_json(selection)\n            check_metadata(spec,info,chosen,family,outer,ds)\n            marker, cache = folder/\'complete.json\', folder/\'predictions.npz\'\n            use_cache = mode != \'fresh_inference\' and cache.exists() and marker.exists()\n            if use_cache:\n                done = read_json(marker)\n                if done.get(\'outer_subject\') != outer or done.get(\'family\') != family:\n                    raise ValueError(\'Outer-evaluation completion marker differs\')\n                pred = read_prediction_cache(cache,te)\n                kind = \'saved held-out predictions\'\n                hashes[family] = {\'predictions_sha256\':digest(cache)}\n            else:\n                if mode == \'cached_only\':\n                    inventory.append(dict(model=family,status=\'unavailable\',reason=\'Completed prediction cache absent; inference mode can use a saved refit\'))\n                    continue\n                weights = folder/\'refit\'/info[\'file\']\n                if not weights.is_file():\n                    raise ValueError(\'Refitted weights missing\')\n                pred = predict_weights(folder/\'refit\',info,te.x,spec[\'software\'])\n                kind = \'fresh inference from saved weights\'\n                hashes[family] = {\'model_sha256\':digest(weights)}\n            metrics = cycle_metrics(te.y,pred)\n            a = te.y.reshape(len(te.y),-1).astype(float)\n            b = pred.reshape(len(pred),-1).astype(float)\n            a -= a.mean(axis=1,keepdims=True)\n            b -= b.mean(axis=1,keepdims=True)\n            denominator = np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)\n            metrics[\'waveform_r\'] = np.divide((a*b).sum(axis=1),denominator,\n                out=np.full(len(a),np.nan),where=denominator>1e-12)\n            metrics.insert(0,\'cycle_id\',te.cycles.cycle_id.to_numpy(str))\n            metrics.insert(0,\'participant_id\',outer)\n            metrics.insert(0,\'model\',family)\n            metrics.insert(0,\'task\',te.task)\n            metrics[\'prediction_source\'] = kind\n            tables.append(metrics)\n            summary.append(dict(task=te.task,participant_id=outer,model=family,n_participants=1,\n                n_cycles=len(te.y),mae=float(metrics.mae.mean()),rmse=float(metrics.rmse.mean()),\n                nrmse=float(metrics.nrmse.mean()),nrmse_percent=float(metrics.nrmse.mean()*100),\n                mean_cycle_waveform_r=float(metrics.waveform_r.mean()),\n                undefined_nrmse_cycles=int(metrics.nrmse.isna().sum()),prediction_source=kind,\n                uncertainty=\'No participant-level confidence interval from one participant\'))\n            predictions[family] = pred\n            hashes[family].update(model_metadata_sha256=digest(descriptor),selection_sha256=digest(selection))\n            np.savez_compressed(output/f\'{family}_heldout_predictions.npz\',\n                cycle_ids=te.cycles.cycle_id.to_numpy(str),measured=te.y,predicted=pred)\n            inventory.append(dict(model=family,status=\'evaluated\',reason=kind))\n        except (ValueError,KeyError,OSError,importlib.metadata.PackageNotFoundError) as error:\n            inventory.append(dict(model=family,status=\'not evaluated\',reason=str(error)))\n    inventory = pd.DataFrame(inventory)\n    inventory.to_csv(output/\'model_availability.csv\',index=False)\n    result = pd.DataFrame(summary)\n    if tables:\n        pd.concat(tables,ignore_index=True).to_csv(output/\'heldout_cycle_metrics.csv\',index=False)\n        result.to_csv(output/\'heldout_participant_metrics.csv\',index=False)\n        if \'mean_waveform\' in set(result.model):\n            reference = result.set_index(\'model\').loc[\'mean_waveform\']\n            comparisons=[]\n            for _, row in result.iterrows():\n                if row.model == \'mean_waveform\': continue\n                comparisons.append(dict(model=row.model,reference=\'mean_waveform\',n_participants=1,\n                    rmse_improvement=reference.rmse-row.rmse,\n                    nrmse_percentage_point_improvement=reference.nrmse_percent-row.nrmse_percent,\n                    interpretation=\'Positive favors model; descriptive result for one held-out participant\'))\n            pd.DataFrame(comparisons).to_csv(output/\'descriptive_baseline_comparisons.csv\',index=False)\n    (output/\'evaluation_provenance.json\').write_text(json.dumps({\n        \'experiment\':str(root),\'task\':ds.task,\'held_out_participant\':outer,\n        \'n_test_participants\':1,\'test_cycle_count\':len(te.y),\'mode\':mode,\n        \'prespecification_sha256\':digest(root/\'prespecification.json\'),\'artifacts\':hashes,\n        \'notes\':[\'No training or tuning performed\',\'No external cohort loaded\',\n                 \'No selected_deep winner declared from these test results\',\n                 \'NRMSE = cycle RMSE / measured full-cycle range; participant result = mean of defined cycle values\',\n                 \'Cycles are repeated observations within one participant; no cohort inference or significance tests\']},indent=2))\n    return inventory,result,predictions,te\n\ndef make_plots(summary, predictions, te, output):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    from matplotlib.ticker import PercentFormatter\n    from math import ceil\n    output=Path(output)\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    labels={\'mean_waveform\':\'Mean waveform\',\'ridge\':\'Ridge\',\'phase_only\':\'Phase only\',\n            \'shallow_mlp\':\'Shallow MLP\',\'lstm\':\'LSTM\',\'gru\':\'GRU\',\'conv1d_lstm\':\'Conv1D-LSTM\'}\n    order=[m for m in FAMILIES if m in predictions]\n    if not order: return []\n    data=summary.set_index(\'model\').loc[order]\n    colors=[\'#6C7A89\' if m not in (\'lstm\',\'gru\',\'conv1d_lstm\') else {\'lstm\':\'#0072B2\',\'gru\':\'#D58A00\',\'conv1d_lstm\':\'#009E73\'}[m] for m in order]\n    subtitle=f\'{te.task} | held-out participant {te.subjects[0]} | {len(te.y)} cycles | n = 1 participant\'\n    fig,axes=plt.subplots(1,3,figsize=(15,4.8))\n    for ax,metric,title in zip(axes,[\'mae\',\'rmse\',\'nrmse_percent\'],[\'Mean cycle MAE\',\'Mean cycle RMSE\',\'Mean cycle NRMSE (%)\']):\n        values=data[metric].to_numpy()\n        ax.barh(range(len(order)),values,color=colors)\n        ax.set_yticks(range(len(order)),[labels[m] for m in order]);ax.invert_yaxis()\n        ax.set_xlabel(\'Target units\' if metric!=\'nrmse_percent\' else \'Percent of measured cycle range\')\n        ax.set_title(title);ax.set_xlim(left=0);ax.grid(axis=\'x\',alpha=.15)\n    fig.suptitle(\'Saved-model held-out errors\\n\'+subtitle,fontsize=14)\n    fig.text(.02,.015,\'Descriptive participant-level results. No cohort confidence intervals. Physical target units require source verification.\',fontsize=9)\n    fig.tight_layout(rect=(0,.07,1,.89))\n    fig.savefig(output/\'Heldout_Model_Errors.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Model_Errors.pdf\');plt.close(fig)\n    phase=np.linspace(0,100,te.y.shape[1]);truth=te.y[:,:,0]\n    fig,axes=plt.subplots(ceil(len(order)/3),3,figsize=(15,3.5*ceil(len(order)/3)),squeeze=False,sharex=True,sharey=True)\n    for ax,family,color in zip(axes.flat,order,colors):\n        low,high=np.quantile(truth,[.25,.75],axis=0)\n        ax.fill_between(phase,low,high,color=\'#ADB5BD\',alpha=.25,label=\'Measured cycle IQR\')\n        ax.plot(phase,truth.mean(axis=0),color=\'black\',lw=1.8,label=\'Measured mean\')\n        ax.plot(phase,predictions[family][:,:,0].mean(axis=0),color=color,lw=1.8,label=\'Predicted mean\')\n        ax.set_title(labels[family]);ax.set_xlabel(\'Normalized gait phase (%)\');ax.set_ylabel(\'GRF (target units)\')\n        ax.grid(alpha=.15)\n    for ax in list(axes.flat)[len(order):]:ax.set_visible(False)\n    handles,legend=axes.flat[0].get_legend_handles_labels()\n    fig.legend(handles,legend,loc=\'lower center\',ncol=3,frameon=False)\n    fig.suptitle(\'Measured and predicted waveforms\\n\'+subtitle,fontsize=14)\n    fig.tight_layout(rect=(0,.07,1,.90))\n    fig.savefig(output/\'Heldout_Waveforms.png\',dpi=180)\n    fig.savefig(output/\'Heldout_Waveforms.pdf\');plt.close(fig)\n    k=min(3,len(truth));fig,axes=plt.subplots(k,1,figsize=(11,3*k),squeeze=False,sharex=True)\n    for i,ax in enumerate(axes[:,0]):\n        ax.plot(phase,truth[i],color=\'black\',lw=2.2,label=\'Measured\')\n        for j,(family,color) in enumerate(zip(order,colors)):\n            ax.plot(phase,predictions[family][i,:,0],color=color,ls=[\'--\',\':\',\'-.\',\'--\',\'-\',\'-\',\'-\'][FAMILIES.index(family)],lw=1.4,label=labels[family])\n        ax.set_title(str(te.cycles.cycle_id.iloc[i]));ax.set_ylabel(\'GRF (target units)\');ax.grid(alpha=.15)\n    axes[-1,0].set_xlabel(\'Normalized gait phase (%)\')\n    fig.legend(*axes[0,0].get_legend_handles_labels(),loc=\'lower center\',ncol=4,frameon=False)\n    fig.suptitle(\'First available cycles in source order; no selection by prediction quality\\n\'+subtitle,fontsize=12)\n    fig.tight_layout(rect=(0,.10,1,.91))\n    fig.savefig(output/\'Heldout_Example_Cycles.png\',dpi=180);plt.close(fig)\n    return [output/name for name in [\'Heldout_Model_Errors.png\',\'Heldout_Waveforms.png\',\'Heldout_Example_Cycles.png\']]\n', 'coordinate_models.py': '"""Subject-wise model-family selection and four explicit comparator families."""\nfrom pathlib import Path\nimport json\nimport warnings\nimport numpy as np\nimport pandas as pd\nimport joblib\nfrom sklearn.linear_model import Ridge\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.neural_network import MLPRegressor\nfrom sklearn.exceptions import ConvergenceWarning\nfrom protocol import cycle_metrics, stable_seed, scale_from_training, write_json, read_json, parameter_hash\n\nBASELINES = (\'mean_waveform\', \'ridge\', \'phase_only\', \'shallow_mlp\')\nDEEP = (\'lstm\', \'gru\', \'conv1d_lstm\')\n\ndef participant_mean(y, groups):\n    return np.mean([y[groups == g].mean(axis=0) for g in np.unique(groups)], axis=0)\n\ndef fourier_basis(timesteps, harmonics):\n    # Endpoints include 0 and 100% consistently with the supplied waveforms.\n    phase = np.linspace(0, 1, timesteps)\n    return np.column_stack([np.ones(timesteps)] +\n        [f(2 * np.pi * k * phase) for k in range(1, harmonics + 1) for f in (np.sin, np.cos)])\n\ndef baseline_grid(family, smoke=False):\n    if family == \'mean_waveform\':\n        return [{}]\n    if family == \'ridge\':\n        return [{\'alpha\': v} for v in ([1.] if smoke else [0.01, 0.1, 1., 10., 100.])]\n    if family == \'phase_only\':\n        return [{\'harmonics\': v} for v in ([3] if smoke else [1, 3, 5, 10])]\n    if family == \'shallow_mlp\':\n        return [{\'units\': u, \'alpha\': a, \'max_iter\': 30 if smoke else 1000}\n                for u in ([4] if smoke else [16, 32, 64])\n                for a in ([0.01] if smoke else [0.0001, 0.01])]\n    raise ValueError(family)\n\ndef fit_baseline(family, x, y, groups, parameters, seed):\n    obj = {\'family\': family, \'shape\': list(y.shape[1:]), \'parameters\': parameters}\n    if family == \'mean_waveform\':\n        obj[\'waveform\'] = participant_mean(y, groups)\n    elif family == \'phase_only\':\n        basis = fourier_basis(y.shape[1], parameters[\'harmonics\'])\n        # Equal participant weighting. Uses phase alone, never ankle positions.\n        coef = np.linalg.lstsq(basis, participant_mean(y, groups), rcond=None)[0]\n        obj[\'waveform\'] = basis @ coef\n    else:\n        flat = x.reshape(len(x), -1)\n        scaler = StandardScaler().fit(flat)  # Training rows only.\n        z = scaler.transform(flat)\n        target = y.reshape(len(y), -1)\n        if family == \'ridge\':\n            model = Ridge(alpha=parameters[\'alpha\'])\n            weights = np.array([1. / np.sum(groups == g) for g in groups])\n            weights *= len(weights) / weights.sum()\n            model.fit(z, target, sample_weight=weights)\n            obj[\'fit_weighting\'] = \'equal participants\'\n        elif family == \'shallow_mlp\':\n            # One hidden layer; training-loss optimizer convergence, NO random validation split.\n            model = MLPRegressor(hidden_layer_sizes=(parameters[\'units\'],), activation=\'relu\',\n                                 solver=\'lbfgs\', alpha=parameters[\'alpha\'], max_iter=parameters[\'max_iter\'],\n                                 max_fun=50000, early_stopping=False, random_state=seed)\n            with warnings.catch_warnings(record=True) as caught:\n                warnings.simplefilter(\'always\', ConvergenceWarning)\n                model.fit(z, target)\n            obj[\'convergence_warnings\'] = [str(w.message) for w in caught]\n            obj[\'fit_weighting\'] = \'equal cycles; validation score remains equal participants\'\n        else:\n            raise ValueError(family)\n        obj.update(model=model, scaler=scaler)\n    return obj\n\ndef predict_baseline(obj, x):\n    if \'waveform\' in obj:\n        return np.repeat(obj[\'waveform\'][None], len(x), axis=0)\n    z = obj[\'scaler\'].transform(x.reshape(len(x), -1))\n    return obj[\'model\'].predict(z).reshape((len(x), *obj[\'shape\']))\n\ndef backend(family, ds, cfg):\n    from models import TensorFlowBackend\n    return TensorFlowBackend(family, ds.x.shape[1], cfg[\'search_profile\'], cfg[\'epochs\'],\n                             cfg[\'patience\'], cfg[\'batch_size\'], cfg[\'verbose\'])\n\ndef cv_candidate(ds, family, parameters, cfg, directory):\n    """Only the development Dataset is passed; outer/external arrays are absent."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    signature = parameter_hash(parameters)\n    rows = []\n    for val in ds.subjects:\n        path = directory / f\'validation_{val}.json\'\n        if path.exists():\n            row = read_json(path)\n            if row[\'parameter_hash\'] != signature or row[\'data_fingerprint\'] != ds.fingerprint:\n                raise ValueError(\'Candidate cache changed\')\n            rows.append(row)\n            continue\n        tr, va = ds.groups != val, ds.groups == val\n        if not tr.any():\n            raise ValueError(\'At least two development participants are required\')\n        seed = stable_seed(cfg[\'seed\'], ds.task, family, val, \'inner\')\n        print(f\'    {family}: validation {val}\', flush=True)\n        if family in DEEP:\n            b = backend(family, ds, cfg)\n            (xt, xv), _ = scale_from_training(ds.x[tr], ds.x[va], enabled=cfg[\'scale_input\'])\n            fit = b.fit_inner(xt, ds.y[tr], xv, ds.y[va], parameters, seed, directory / f\'history_{val}.csv\')\n            prediction, epoch = fit[\'prediction\'], fit[\'best_epoch\']\n            convergence = []\n        else:\n            model = fit_baseline(family, ds.x[tr], ds.y[tr], ds.groups[tr], parameters, seed)\n            prediction, epoch = predict_baseline(model, ds.x[va]), None\n            convergence = model.get(\'convergence_warnings\', [])\n        metric = cycle_metrics(ds.y[va], np.asarray(prediction))\n        row = {\'validation_subject\': val, \'training_subjects\': sorted(set(ds.groups[tr])),\n               \'n_validation_cycles\': int(va.sum()), \'parameter_hash\': signature,\n               \'data_fingerprint\': ds.fingerprint, \'mae\': float(metric.mae.mean()),\n               \'rmse\': float(metric.rmse.mean()), \'best_epoch\': epoch, \'convergence_warnings\': convergence}\n        write_json(path, row)\n        rows.append(row)\n    score = float(np.mean([r[\'mae\'] for r in rows]))\n    if not np.isfinite(score):\n        raise ValueError(\'Nonfinite full subject-wise validation objective\')\n    result = {\'family\': family, \'parameters\': parameters, \'inner_subject_mae\': score, \'folds\': rows,\n              \'epochs\': max(1, int(np.ceil(np.median([r[\'best_epoch\'] for r in rows])))) if family in DEEP else None}\n    write_json(directory / \'summary.json\', result)\n    return result\n\ndef tune_family(ds, family, cfg, directory):\n    directory = Path(directory)\n    done = directory / \'selected.json\'\n    if done.exists():\n        return read_json(done)\n    directory.mkdir(parents=True, exist_ok=True)\n    if family not in DEEP:\n        candidates = [cv_candidate(ds, family, p, cfg, directory / f\'candidate_{i:03d}\')\n                      for i, p in enumerate(baseline_grid(family, cfg.get(\'smoke\', False)))]\n        selected = min(candidates, key=lambda r: (r[\'inner_subject_mae\'], parameter_hash(r[\'parameters\'])))\n    else:\n        from models import training_dependencies, search_space\n        tf, kt = training_dependencies()\n        class SubjectTuner(kt.Tuner):\n            def run_trial(self, trial, *args, **kwargs):\n                try:\n                    result = cv_candidate(ds, family, dict(trial.hyperparameters.values), cfg,\n                                          directory / \'candidates\' / trial.trial_id)\n                except (ValueError, tf.errors.ResourceExhaustedError) as exc:\n                    write_json(directory / \'candidates\' / trial.trial_id / \'failure.json\', {\'error\': str(exc)})\n                    raise kt.errors.FailedTrialError(str(exc)) from exc\n                return {\'inner_subject_mae\': result[\'inner_subject_mae\']}\n        hp = search_space(family, cfg[\'search_profile\'])\n        oracle = kt.oracles.BayesianOptimizationOracle(objective=kt.Objective(\'inner_subject_mae\', \'min\'),\n            max_trials=cfg[\'max_trials\'], num_initial_points=min(3, cfg[\'max_trials\']),\n            hyperparameters=hp, allow_new_entries=False, tune_new_entries=False,\n            seed=stable_seed(cfg[\'seed\'], ds.task, family, \'oracle\'), max_consecutive_failed_trials=cfg[\'max_trials\'] + 1)\n        tuner = SubjectTuner(oracle=oracle, hypermodel=None, directory=str(directory), project_name=\'tuner\', overwrite=False)\n        tuner.search()\n        best = tuner.oracle.get_best_trials(1)\n        if not best or best[0].score is None or not np.isfinite(best[0].score):\n            raise ValueError(f\'No complete successful candidate for {family}\')\n        selected = read_json(directory / \'candidates\' / best[0].trial_id / \'summary.json\')\n    write_json(done, selected)\n    return selected\n\ndef fit_frozen_model(ds, selected, cfg, directory):\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    descriptor = directory / \'model.json\'\n    if descriptor.exists():\n        return read_json(descriptor)\n    family = selected[\'family\']\n    seed = stable_seed(cfg[\'seed\'], ds.task, family, \'final_refit\')\n    if family in DEEP:\n        b = backend(family, ds, cfg)\n        (x,), scaler = scale_from_training(ds.x, enabled=cfg[\'scale_input\'])\n        model = b.fit_final(x, ds.y, selected[\'parameters\'], seed, selected[\'epochs\'], directory / \'history.csv\')\n        b.save(model, directory / \'model.keras\')\n        b.clear()\n        result = {\'family\': family, \'file\': \'model.keras\', \'input_scaler\': scaler}\n    else:\n        model = fit_baseline(family, ds.x, ds.y, ds.groups, selected[\'parameters\'], seed)\n        joblib.dump(model, directory / \'model.joblib\')\n        result = {\'family\': family, \'file\': \'model.joblib\',\n                  \'convergence_warnings\': model.get(\'convergence_warnings\', [])}\n    result.update(parameters=selected[\'parameters\'], epochs=selected[\'epochs\'], seed=seed,\n                  n_training_participants=len(ds.subjects), n_training_cycles=len(ds.x),\n                  training_participants=ds.subjects, data_fingerprint=ds.fingerprint)\n    write_json(descriptor, result)\n    return result\n\ndef predict_saved(directory, x):\n    directory = Path(directory)\n    info = read_json(directory / \'model.json\')\n    if info[\'family\'] in DEEP:\n        from models import training_dependencies\n        tf, _ = training_dependencies()\n        scaler = info[\'input_scaler\']\n        z = (x - np.array(scaler[\'mean\'])[None, None]) / np.array(scaler[\'std\'])[None, None]\n        model = tf.keras.models.load_model(directory / info[\'file\'], compile=False)\n        try:\n            return np.asarray(model.predict(z, verbose=0))\n        finally:\n            del model\n            tf.keras.backend.clear_session()\n    # Only load models created by this notebook, whose file hashes are frozen.\n    return np.asarray(predict_baseline(joblib.load(directory / info[\'file\']), x))\n\ndef mean_ensemble(predictions):\n    """Average predictions BEFORE errors: models are not independent participants."""\n    values = [np.asarray(p) for p in predictions]\n    if not values or any(v.shape != values[0].shape or not np.isfinite(v).all() for v in values):\n        raise ValueError(\'Ensemble members require identical finite cycle/time/component arrays\')\n    return np.mean(np.stack(values), axis=0)\n', 'coordinate_checks.py': '"""Coordinate ablations and held-out nuisance tests, isolated from the original run."""\nfrom dataclasses import replace\nfrom pathlib import Path\nimport hashlib\nimport importlib.metadata\nimport json\nimport traceback\nimport numpy as np\nimport pandas as pd\nfrom protocol import write_json, read_json, cycle_metrics\nfrom revision_data import coordinate_dataset, gait_basis, AXIS\nfrom evaluate_saved import check_metadata, read_prediction_cache, predict_weights, digest\n\nMODES = (\'absolute\',\'translation\',\'pelvis\',\'aligned\',\'body\',\'combined\')\nFAMILIES = (\'mean_waveform\',\'ridge\',\'phase_only\',\'shallow_mlp\',\'lstm\',\'gru\',\'conv1d_lstm\')\nTRAINING_KEYS = (\'seed\',\'search_profile\',\'max_trials\',\'epochs\',\'patience\',\'batch_size\',\'verbose\',\'scale_input\',\'smoke\')\n\ndef canonical_hash(obj):\n    return hashlib.sha256(json.dumps(obj,sort_keys=True,allow_nan=False).encode()).hexdigest()\n\ndef versions():\n    out={}\n    for name in [\'numpy\',\'pandas\',\'scikit-learn\',\'tensorflow\',\'keras\',\'keras-tuner\',\'joblib\']:\n        try: out[name]=importlib.metadata.version(name)\n        except importlib.metadata.PackageNotFoundError: out[name]=None\n    return out\n\ndef require_versions(reference, family):\n    names=[\'numpy\',\'pandas\']+([\'tensorflow\',\'keras\',\'keras-tuner\'] if family in FAMILIES[-3:] else [\'scikit-learn\',\'joblib\'])\n    for name in names:\n        if not reference.get(name) or versions().get(name)!=reference[name]:\n            raise ValueError(f\'{name}: use training version {reference.get(name)}; found {versions().get(name)}. Run the setup cell and restart if requested.\')\n\ndef verify_folds(selected, train):\n    folds=selected.get(\'folds\',[])\n    names=[r[\'validation_subject\'] for r in folds]\n    if sorted(names)!=train.subjects or len(names)!=len(set(names)):\n        raise ValueError(\'Selection does not contain one complete inner fold for every training participant\')\n    for row in folds:\n        expected=set(train.subjects)-{row[\'validation_subject\']}\n        if set(row[\'training_subjects\'])!=expected or len(row[\'training_subjects\'])!=len(expected):\n            raise ValueError(\'Invalid or overlapping inner training/validation participants\')\n        if row[\'data_fingerprint\']!=train.fingerprint or not np.isfinite(row[\'mae\']):\n            raise ValueError(\'Invalid inner fold fingerprint or score\')\n\ndef original_model(source, ds, outer, family):\n    source=Path(source)\n    spec=read_json(source/\'prespecification.json\')\n    folder=source/\'nested\'/outer/family\n    info=read_json(folder/\'refit/model.json\')\n    selected=read_json(folder/\'selection/selected.json\')\n    check_metadata(spec,info,selected,family,outer,ds)\n    verify_folds(selected,ds.subset([s for s in ds.subjects if s!=outer]))\n    return folder,info,selected,spec\n\ndef verify_original_source(source, cohort, code_dir):\n    spec=read_json(Path(source)/\'prespecification.json\')\n    ds=coordinate_dataset(cohort,\'absolute\')\n    if spec[\'data_fingerprint\']!=ds.fingerprint or spec[\'task\']!=ds.task:\n        raise ValueError(\'Original data fingerprint/task differs from the saved experiment\')\n    if sorted(spec[\'subjects\'])!=ds.subjects or spec[\'config\'].get(\'smoke\'):\n        raise ValueError(\'Saved participant list or research/smoke setting is invalid\')\n    for name in [\'data.py\',\'protocol.py\',\'models.py\',\'revision_data.py\',\'revision_models.py\']:\n        if digest(Path(code_dir)/name)!=spec[\'sources\'].get(name):\n            raise ValueError(f\'Original source mismatch: {name}\')\n    return spec\n\ndef make_templates(cohort, folder):\n    """Create missing templates, never overwrite user-supplied metadata."""\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    paths={}\n    paths[\'cycle_id_reference.csv\']=cohort.ds.cycles[[\'cycle_id\',\'participant_id\']]\n    paths[\'participant_body_scale.csv\']=pd.DataFrame({\'participant_id\':cohort.ds.subjects,\n        \'body_scale_m\':[\'\']*len(cohort.ds.subjects)})\n    for name,frame in paths.items():\n        if not (folder/name).exists():frame.to_csv(folder/name,index=False)\n    meta=dict(task=cohort.ds.task,input_length_unit=\'unverified\',units_verified=False,\n        canonical_axis_order=[\'AP\',\'ML\',\'Vertical\'],frame_and_axes_verified=False,\n        cycle_alignment_verified=False,targets_already_in_gait_frame_verified=False,\n        body_scale_definition=\'\',body_scale_source=\'\',direction_source=\'\',\n        direction_independent_of_grf_verified=False,\n        notes=\'Do not change verification flags unless supported by the source data/protocol.\')\n    if not (folder/\'coordinate_metadata.json\').exists():write_json(folder/\'coordinate_metadata.json\',meta)\n    text=\'\'\'OPTIONAL ANATOMICAL INPUTS - DO NOT INVENT MISSING ARRAYS\n\nAll cycles must match cycle_id_reference.csv exactly by ID. The loader reorders by ID.\nCreate coordinate_inputs.npz (numpy.savez_compressed) with cycle_ids: Unicode [N].\nOptional real, synchronized arrays:\n  ankle_xyz: [N,100,3], in original laboratory length units and AP,ML,Vertical order.\n  pelvis_xyz: [N,100,3], same time points, frame, units and origin as ankles.\n  direction_xyz: [N,3], nonzero horizontal forward vectors, derived without GRF.\nDo not join separate component files by column number without verifying cycle pairing.\nThe ankle component is checked against the original internal ZIP before use.\nPelvis-only arrays can support component-wise pelvis centering without ankle_xyz.\n\nBody normalization can run on scalar ankle trajectories: complete participant_body_scale.csv\nwith real height OR leg length in metres and one prespecified definition for all participants.\nSet its source/definition and verified original input unit (m/mm) in coordinate_metadata.json.\nDo not fill missing anthropometry with cohort means or estimates.\n\nFor alignment/rotation tests, original scalar GRF targets must already represent gait-frame\ncomponents; verify this in coordinate_metadata.json. If they are laboratory-frame components,\nthis companion cannot reuse the original scalar targets: rebuild a consistently harmonized\n3D kinematics/force cohort before that analysis. This notebook does not rotate targets by guess.\n\nTranslation SD tests require no assumed physical unit: offsets use only outer-training inputs.\nExternal data must not be used to populate or select settings for this internal analysis.\n\'\'\'\n    if not (folder/\'INPUT_GUIDE.txt\').exists():(folder/\'INPUT_GUIDE.txt\').write_text(text)\n\ndef seed_known_heights(folder, heights_m, provenance, confirmed_units=\'unverified\', unit_evidence=\'\'):\n    """Use supplied, explicitly labelled clinical heights; never replace user-entered scales."""\n    folder=Path(folder);meta=read_json(folder/\'coordinate_metadata.json\')\n    body=pd.read_csv(folder/\'participant_body_scale.csv\',dtype=str,keep_default_na=False)\n    if body.body_scale_m.str.strip().eq(\'\').all() and not meta.get(\'body_scale_definition\') and not meta.get(\'body_scale_source\'):\n        if set(body.participant_id).issubset(heights_m):\n            values=body.participant_id.map(heights_m).astype(float)\n            if not np.isfinite(values).all() or (values<=0).any():raise ValueError(\'Invalid supplied clinical height\')\n            body[\'body_scale_m\']=values\n            body.to_csv(folder/\'participant_body_scale.csv\',index=False)\n            meta[\'body_scale_definition\']=\'Participant height: explicitly labelled Height (cm), converted to metres\'\n            meta[\'body_scale_source\']=provenance\n    if confirmed_units not in (\'unverified\',\'m\',\'mm\'):raise ValueError(\'Use unverified, m or mm for CONFIRMED_ANKLE_UNITS\')\n    if confirmed_units!=\'unverified\':\n        if not str(unit_evidence).strip():raise ValueError(\'Document ANKLE_UNITS_EVIDENCE before confirming physical units\')\n        if meta.get(\'units_verified\') and meta.get(\'input_length_unit\')!=confirmed_units:\n            raise ValueError(\'Confirmed units conflict with previously supplied coordinate metadata\')\n        meta.update(input_length_unit=confirmed_units,units_verified=True,unit_evidence=str(unit_evidence))\n    write_json(folder/\'coordinate_metadata.json\',meta)\n\nclass CoordinateInputs:\n    def __init__(self, cohort, folder):\n        self.cohort=cohort; self.ds=cohort.ds; self.folder=Path(folder)\n        self.meta=read_json(self.folder/\'coordinate_metadata.json\');self.arrays={}\n        if self.meta.get(\'task\')!=self.ds.task:raise ValueError(\'Coordinate metadata task differs\')\n        self.body=None\n        files=[\'coordinate_metadata.json\']\n        path=self.folder/\'coordinate_inputs.npz\'\n        if path.exists():\n            with np.load(path,allow_pickle=False) as a:\n                raw={k:a[k] for k in a.files}\n            ids=raw.pop(\'cycle_ids\').astype(str)\n            expected=self.ds.cycles.cycle_id.to_numpy(str)\n            if len(set(ids))!=len(ids) or set(ids)!=set(expected):\n                raise ValueError(\'Anatomical NPZ must contain exactly all original cycle IDs\')\n            order=pd.Index(ids).get_indexer(expected)\n            for key,arr in raw.items():\n                if key not in (\'ankle_xyz\',\'pelvis_xyz\',\'direction_xyz\'):raise ValueError(f\'Unknown NPZ field: {key}\')\n                shape=(len(ids),3) if key==\'direction_xyz\' else (len(ids),100,3)\n                if arr.shape!=shape or not np.issubdtype(arr.dtype,np.number) or not np.isfinite(arr).all():\n                    raise ValueError(f\'Invalid {key}: expected finite {shape}\')\n                self.arrays[key]=arr[order].astype(float)\n            files.append(path.name)\n        path=self.folder/\'participant_body_scale.csv\'\n        if path.exists():\n            body=pd.read_csv(path,dtype={\'participant_id\':str},keep_default_na=False)\n            if set([\'participant_id\',\'body_scale_m\'])-set(body):raise ValueError(\'Body-scale CSV columns missing\')\n            if body.participant_id.duplicated().any() or set(body.participant_id)!=set(self.ds.subjects):\n                raise ValueError(\'Body-scale participant IDs must match this task exactly\')\n            vals=pd.to_numeric(body.body_scale_m,errors=\'coerce\')\n            if np.isfinite(vals).all() and (vals>0).all():\n                mapping=dict(zip(body.participant_id,vals));self.body=np.array([mapping[g] for g in self.ds.groups])\n            files.append(path.name)\n        self.fingerprint=canonical_hash({name:digest(self.folder/name) for name in files})\n\n    def check_frame(self):\n        if self.meta.get(\'canonical_axis_order\')!=[\'AP\',\'ML\',\'Vertical\'] or not self.meta.get(\'frame_and_axes_verified\') or not self.meta.get(\'cycle_alignment_verified\'):\n            raise ValueError(\'Verify anatomical axes, common frame and cycle/time alignment in coordinate_metadata.json\')\n        if \'ankle_xyz\' in self.arrays:\n            axis=AXIS[self.ds.task.split(\'_\')[0]]\n            if not np.allclose(self.arrays[\'ankle_xyz\'][:,:,axis:axis+1],self.ds.x,rtol=1e-5,atol=1e-5):\n                raise ValueError(\'Anatomical ankle component differs from original inputs: frame, units or cycle pairing mismatch\')\n\n    def require(self,mode,rotation=False):\n        if mode not in MODES:raise ValueError(\'Unknown coordinate mode\')\n        if mode in (\'pelvis\',\'aligned\',\'combined\') or rotation:\n            self.check_frame()\n        if mode in (\'pelvis\',\'combined\') and \'pelvis_xyz\' not in self.arrays:\n            raise ValueError(\'Synchronized pelvis_xyz is missing\')\n        if mode in (\'aligned\',\'combined\') or rotation:\n            if \'ankle_xyz\' not in self.arrays or \'direction_xyz\' not in self.arrays:\n                raise ValueError(\'Verified ankle_xyz and force-independent direction_xyz are required\')\n            if not self.meta.get(\'direction_independent_of_grf_verified\') or not self.meta.get(\'direction_source\'):\n                raise ValueError(\'Document and verify the force-independent walking-direction source\')\n            gait_basis(self.arrays[\'direction_xyz\'])\n            if not self.meta.get(\'targets_already_in_gait_frame_verified\'):\n                raise ValueError(\'Verify that unchanged GRF targets are already gait-frame components before alignment or rotation tests\')\n        if mode in (\'body\',\'combined\'):\n            if self.body is None:raise ValueError(\'Complete verified participant-linked body_scale_m values\')\n            if not self.meta.get(\'units_verified\') or self.meta.get(\'input_length_unit\') not in (\'m\',\'mm\'):\n                raise ValueError(\'Verify original input length units (m/mm) before body normalization\')\n            if not self.meta.get(\'body_scale_definition\') or not self.meta.get(\'body_scale_source\'):\n                raise ValueError(\'Prespecify and document a common height or leg-length definition\')\n\n    def dataset(self,mode,offset=None,yaw=0.):\n        self.require(mode,rotation=bool(yaw))\n        if mode==\'absolute\' and offset is None and yaw==0:\n            return coordinate_dataset(self.cohort,\'absolute\')\n        axis=AXIS[self.ds.task.split(\'_\')[0]]\n        xyz=self.arrays.get(\'ankle_xyz\');xyz=None if xyz is None else xyz.copy()\n        pelvis=self.arrays.get(\'pelvis_xyz\');pelvis=None if pelvis is None else pelvis.copy()\n        direction=self.arrays.get(\'direction_xyz\');direction=None if direction is None else direction.copy()\n        # Unperturbed scalar features retain the exact original values.\n        x=self.ds.x.astype(float).copy()\n        if yaw:\n            theta=np.deg2rad(yaw);c,s=np.cos(theta),np.sin(theta)\n            rotation=np.array([[c,-s,0.],[s,c,0.],[0.,0.,1.]])\n            xyz=xyz@rotation.T;direction=direction@rotation.T\n            if pelvis is not None:pelvis=pelvis@rotation.T\n            x=xyz[:,:,axis:axis+1]\n        if offset is not None:\n            offset=np.asarray(offset,float)\n            if offset.shape!=(3,) or not np.isfinite(offset).all():raise ValueError(\'Invalid translation vector\')\n            x=x+offset[axis]\n            if xyz is not None:xyz=xyz+offset\n            if pelvis is not None:pelvis=pelvis+offset\n        if mode in (\'translation\',\'body\'):\n            x=x-x[:,:1,:]\n        elif mode==\'pelvis\':x=x-pelvis[:,:,axis:axis+1]\n        elif mode in (\'aligned\',\'combined\'):\n            centered=xyz-pelvis if mode==\'combined\' else xyz-xyz[:,:1,:]\n            x=np.einsum(\'nti,nij->ntj\',centered,gait_basis(direction))[:,:,axis:axis+1]\n        if mode in (\'body\',\'combined\'):\n            factor=1000. if self.meta[\'input_length_unit\']==\'mm\' else 1.\n            x=x/(self.body[:,None,None]*factor)\n        fp=canonical_hash({\'original\':self.ds.fingerprint,\'supplement\':self.fingerprint,\'mode\':mode,\n                           \'offset\':None if offset is None else offset.tolist(),\'yaw\':float(yaw)})\n        return replace(self.ds,x=x.astype(\'float32\'),fingerprint=fp)\n\ndef metric_tables(ds,pred,outer,mode,family,scenario=\'unperturbed\',reference=None):\n    t=cycle_metrics(ds.y,pred)\n    a=ds.y.reshape(len(ds.y),-1).astype(float);b=pred.reshape(len(pred),-1).astype(float)\n    a=a-a.mean(axis=1,keepdims=True);b=b-b.mean(axis=1,keepdims=True)\n    d=np.linalg.norm(a,axis=1)*np.linalg.norm(b,axis=1)\n    t[\'waveform_r\']=np.divide(np.sum(a*b,axis=1),d,out=np.full(len(a),np.nan),where=d>1e-12)\n    t[\'nrmse_percent\']=100*t.nrmse\n    for key,value in dict(task=ds.task,participant_id=outer,coordinate=mode,model=family,scenario=scenario).items():t[key]=value\n    t[\'cycle_id\']=ds.cycles.cycle_id.to_numpy(str)\n    if reference is not None:\n        ref=cycle_metrics(ds.y,reference)\n        t[\'rmse_change\']=t.rmse-ref.rmse\n        t[\'prediction_change_rmse\']=np.sqrt(np.mean((pred.astype(float)-reference.astype(float))**2,axis=(1,2)))\n    keys=[\'task\',\'participant_id\',\'coordinate\',\'model\',\'scenario\']\n    measures=[\'rmse\',\'mae\',\'nrmse_percent\',\'waveform_r\']+([\'rmse_change\',\'prediction_change_rmse\'] if reference is not None else [])\n    summary=t.groupby(keys,dropna=False)[measures].mean().reset_index()\n    summary[\'n_cycles\']=len(t);summary[\'undefined_nrmse_cycles\']=int(t.nrmse.isna().sum())\n    return t,summary\n\ndef bind(folder,signature):\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True)\n    path=folder/\'case_manifest.json\'\n    if path.exists() and read_json(path)!=signature:\n        raise ValueError(\'Case settings/data/code/software changed; choose a new RUN_ID. Existing results were not overwritten.\')\n    if not path.exists():write_json(path,signature)\n\ndef case_key(task,outer,mode,family,stage,scenario=\'unperturbed\'):\n    return \'/\'.join([task,outer,mode,family,stage,scenario])\n\ndef record(root,key,status,reason=\'\'):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    path=root/\'status.json\';items=read_json(path) if path.exists() else {}\n    items[key]={\'case\':key,\'status\':status,\'reason\':str(reason)}\n    write_json(path,items);pd.DataFrame(items.values()).to_csv(root/\'status.csv\',index=False)\n    print(status.upper(),key, str(reason)[:240],flush=True)\n\ndef current_sources():\n    folder=Path(__file__).parent\n    return {p.name:digest(p) for p in folder.glob(\'*.py\')}\n\ndef execute_case(inputs,source,root,outer,mode,family,strategy,cfg):\n    """Return saved model location, validated prediction, case folder and software."""\n    from coordinate_models import tune_family,fit_frozen_model,predict_saved\n    ds=inputs.dataset(mode);te=ds.subset([outer]);train=ds.subset([s for s in ds.subjects if s!=outer])\n    if len(train.subjects)<2:raise ValueError(\'Need at least three participants for nested LOSO\')\n    case=Path(root)/\'cases\'/ds.task/outer/mode/family\n    absds=inputs.dataset(\'absolute\')\n    original=None\n    if mode==\'absolute\' or strategy==\'frozen_selection\':\n        try:original=original_model(source,absds,outer,family)\n        except FileNotFoundError:\n            if strategy==\'frozen_selection\':raise ValueError(\'No completed original inner selection/refit for this model and outer participant; use nested_reselect to train it\')\n    signature={\'dataset\':ds.fingerprint,\'outer\':outer,\'mode\':mode,\'family\':family,\'strategy\':strategy,\n               \'configuration\':{k:cfg[k] for k in TRAINING_KEYS},\'sources\':current_sources(),\'software\':versions()}\n    if original:\n        folder,info,selection,spec=original\n        signature[\'source_model_metadata\']=digest(folder/\'refit/model.json\')\n        signature[\'source_selection\']=digest(folder/\'selection/selected.json\')\n        signature[\'source_prespecification\']=digest(Path(source)/\'prespecification.json\')\n        if (folder/\'predictions.npz\').exists():signature[\'source_predictions\']=digest(folder/\'predictions.npz\')\n    bind(case,signature)\n    if mode==\'absolute\' and original:\n        folder,info,selection,spec=original\n        marker=read_json(folder/\'complete.json\')\n        if marker.get(\'outer_subject\')!=outer or marker.get(\'family\')!=family:raise ValueError(\'Original completion marker mismatch\')\n        pred=read_prediction_cache(folder/\'predictions.npz\',te)\n        model_path=folder/\'refit\';software=spec[\'software\'];origin=\'original completed outer test\'\n    else:\n        model_path=case/\'refit\';software=versions()\n        if (case/\'complete.json\').exists():\n            if read_json(case/\'complete.json\')[\'prediction_sha256\']!=digest(case/\'predictions.npz\'):\n                raise ValueError(\'Saved coordinate predictions changed after completion\')\n            pred=read_prediction_cache(case/\'predictions.npz\',te)\n            info=read_json(model_path/\'model.json\')\n            chosen=read_json(case/\'selection/selected.json\')\n        else:\n            if strategy==\'frozen_selection\':\n                folder,info,selection,spec=original\n                require_versions(spec[\'software\'],family)\n                chosen={\'family\':family,\'parameters\':selection[\'parameters\'],\'epochs\':selection[\'epochs\'],\n                        \'selection_strategy\':\'fixed from original absolute-coordinate inner LOSO\',\n                        \'source_absolute_inner_subject_mae\':selection[\'inner_subject_mae\'],\n                        \'source_selection_sha256\':digest(folder/\'selection/selected.json\')}\n                write_json(case/\'selection/selected.json\',chosen)\n            else:\n                chosen=tune_family(train,family,cfg,case/\'selection\')\n                verify_folds(chosen,train)\n            info=fit_frozen_model(train,chosen,cfg,model_path)\n            pred=predict_saved(model_path,te.x)\n        if info[\'training_participants\']!=train.subjects or outer in info[\'training_participants\'] or info[\'data_fingerprint\']!=train.fingerprint:\n            raise ValueError(\'Refitted model training participants/data mismatch\')\n        if info[\'parameters\']!=chosen[\'parameters\'] or info[\'epochs\']!=chosen[\'epochs\']:\n            raise ValueError(\'Refitted model differs from frozen selection\')\n        origin=strategy\n    cycles,summary=metric_tables(te,pred,outer,mode,family)\n    cycles.to_csv(case/\'cycle_metrics.csv\',index=False);summary[\'selection_strategy\']=origin\n    summary.to_csv(case/\'participant_metrics.csv\',index=False)\n    np.savez_compressed(case/\'predictions.npz\',cycle_ids=te.cycles.cycle_id.to_numpy(str),measured=te.y,predicted=pred)\n    write_json(case/\'complete.json\',{\'model\':family,\'outer\':outer,\'coordinate\':mode,\'strategy\':origin,\n        \'prediction_sha256\':digest(case/\'predictions.npz\'),\'model_directory\':str(model_path)})\n    return model_path,info,pred,case,software\n\ndef perturb_case(inputs,root,outer,mode,family,model_path,info,pred,case,software,options):\n    """Shift/rotate held-out inputs only; no fitting, no test-driven choice of perturbations."""\n    ds=inputs.dataset(mode);te=ds.subset([outer]);trmask=inputs.ds.groups!=outer\n    axis=AXIS[ds.task.split(\'_\')[0]]\n    scenarios=[]\n    if options[\'translation\']:\n        sd=float(np.std(inputs.ds.x[trmask].astype(float)))\n        if not np.isfinite(sd) or sd<=1e-12:raise ValueError(\'Outer-training input SD is zero; no defined SD perturbation\')\n        for multiple in options[\'translation_sd_multipliers\']:\n            vector=np.zeros(3);vector[axis]=float(multiple)*sd\n            scenarios.append((f\'translation_sd_{multiple:+g}\',vector,0.,{\'translation_sd\':float(multiple),\'source_unit_offset\':float(vector[axis])}))\n    if options[\'rotation\']:\n        try:inputs.require(mode,rotation=True)\n        except ValueError as e:\n            record(root,case_key(ds.task,outer,mode,family,\'perturbation\',\'rotation\'),\'blocked\',str(e))\n        else:\n            for angle in options[\'yaw_degrees\']:\n                scenarios.append((f\'yaw_{angle:+g}\',None,float(angle),{\'yaw_degrees\':float(angle)}))\n    if not scenarios:return\n    model_file=Path(model_path)/info[\'file\']\n    context={\'model_sha256\':digest(model_file),\'original_case\':digest(case/\'case_manifest.json\'),\n             \'metadata\':inputs.fingerprint,\'options\':options,\'software\':versions()}\n    fresh=None\n    for name,offset,yaw,fields in scenarios:\n        key=case_key(ds.task,outer,mode,family,\'perturbation\',name)\n        folder=case/\'perturbations\'/name\n        try:\n            transformed=inputs.dataset(mode,offset=offset,yaw=yaw).subset([outer])\n            bind(folder,{**context,\'scenario\':name,\'fields\':fields})\n            if (folder/\'complete.json\').exists():\n                if read_json(folder/\'complete.json\')[\'prediction_sha256\']!=digest(folder/\'predictions.npz\'):\n                    raise ValueError(\'Saved perturbation predictions changed after completion\')\n                shifted=read_prediction_cache(folder/\'predictions.npz\',transformed)\n            else:\n                if fresh is None:\n                    fresh=predict_weights(model_path,info,te.x,software)\n                    if not np.allclose(fresh,pred,rtol=1e-4,atol=1e-5):\n                        raise ValueError(\'Saved weights do not reproduce the unperturbed prediction cache\')\n                shifted=predict_weights(model_path,info,transformed.x,software)\n            cycle,summary=metric_tables(transformed,shifted,outer,mode,family,name,reference=pred)\n            for col,val in fields.items():summary[col]=val;cycle[col]=val\n            summary[\'translation_definition\']=\'Constant shift along the original task axis, in multiples of pooled outer-training ankle-input SD\'\n            cycle.to_csv(folder/\'cycle_metrics.csv\',index=False);summary.to_csv(folder/\'participant_metrics.csv\',index=False)\n            np.savez_compressed(folder/\'predictions.npz\',cycle_ids=transformed.cycles.cycle_id.to_numpy(str),measured=transformed.y,predicted=shifted)\n            write_json(folder/\'complete.json\',{\'scenario\':name,\'prediction_sha256\':digest(folder/\'predictions.npz\')})\n            record(root,key,\'completed\')\n        except Exception as e:\n            record(root,key,\'failed\',str(e));(case/\'last_perturbation_error.txt\').write_text(traceback.format_exc())\n\ndef run_task(cohort,source,root,supplement,outer_subjects,modes,families,strategy,options,cfg):\n    if strategy not in (\'frozen_selection\',\'nested_reselect\'):raise ValueError(\'Unknown selection strategy\')\n    root=Path(root);source=Path(source)\n    if root.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(root.resolve()):\n        raise ValueError(\'New output and original training directories must be separate\')\n    inputs=CoordinateInputs(cohort,supplement)\n    outers=cohort.ds.subjects if outer_subjects==\'all\' else list(outer_subjects)\n    if set(outers)-set(cohort.ds.subjects) or len(outers)!=len(set(outers)):raise ValueError(\'Unknown/duplicate outer participant\')\n    if set(modes)-set(MODES) or set(families)-set(FAMILIES):raise ValueError(\'Unknown model or coordinate mode\')\n    bind(root,{\'strategy\':strategy,\'coordinate_modes\':modes,\'families\':families,\'perturbations\':options,\n               \'configuration\':{k:cfg[k] for k in TRAINING_KEYS},\'software\':versions(),\'sources\':current_sources()})\n    bind(root/\'input_manifests\'/cohort.ds.task,{\'dataset\':cohort.ds.fingerprint,\'supplement\':inputs.fingerprint})\n    write_json(root/f\'{cohort.ds.task}_requested_scope.json\',dict(task=cohort.ds.task,outer_participants=outers,\n        coordinate_modes=modes,families=families,selection_strategy=strategy,\n        note=\'Default single-outer sensitivity run is not a complete cohort nested-LOSO study.\'))\n    # Do all saved absolute-model tests FIRST, so a long normalized refit cannot hide these outputs.\n    for mode in modes:\n        try:inputs.require(mode)\n        except ValueError as e:\n            record(root,case_key(cohort.ds.task,\'all_requested\',mode,\'all_requested\',\'coordinate\'),\'blocked\',str(e));continue\n        for outer in outers:\n            for family in families:\n                key=case_key(cohort.ds.task,outer,mode,family,\'coordinate\')\n                record(root,key,\'running\')\n                try:\n                    model_path,info,pred,case,software=execute_case(inputs,source,root,outer,mode,family,strategy,cfg)\n                    record(root,key,\'completed\')\n                except Exception as e:\n                    record(root,key,\'failed_or_unavailable\',str(e));continue\n                try:perturb_case(inputs,root,outer,mode,family,model_path,info,pred,case,software,options)\n                except Exception as e:record(root,case_key(cohort.ds.task,outer,mode,family,\'perturbation\'),\'failed\',str(e))\n                summarize(root)\n    return summarize(root)\n\ndef summarize(root):\n    root=Path(root);root.mkdir(parents=True,exist_ok=True)\n    rows=[pd.read_csv(p) for p in (root/\'cases\').glob(\'*/*/*/*/participant_metrics.csv\')]\n    metrics=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()\n    if len(metrics):\n        metrics.to_csv(root/\'coordinate_participant_metrics.csv\',index=False)\n        baseline=metrics.loc[metrics.coordinate==\'absolute\',[\'task\',\'participant_id\',\'model\',\'rmse\',\'nrmse_percent\']]\n        paired=metrics.loc[metrics.coordinate!=\'absolute\'].merge(baseline,on=[\'task\',\'participant_id\',\'model\'],suffixes=(\'\',\'_absolute\'),validate=\'many_to_one\')\n        paired[\'rmse_improvement\']=paired.rmse_absolute-paired.rmse\n        paired[\'nrmse_percentage_point_improvement\']=paired.nrmse_percent_absolute-paired.nrmse_percent\n        paired.to_csv(root/\'paired_coordinate_comparisons.csv\',index=False)\n        counts=metrics.groupby([\'task\',\'coordinate\',\'model\']).agg(n_participants=(\'participant_id\',\'nunique\'),n_cycles=(\'n_cycles\',\'sum\'),mean_participant_rmse=(\'rmse\',\'mean\'),mean_participant_nrmse_percent=(\'nrmse_percent\',\'mean\')).reset_index()\n        counts.to_csv(root/\'coordinate_summary.csv\',index=False)\n    rows=[pd.read_csv(p) for p in (root/\'cases\').glob(\'*/*/*/*/perturbations/*/participant_metrics.csv\')]\n    perturb=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()\n    if len(perturb):perturb.to_csv(root/\'perturbation_participant_metrics.csv\',index=False)\n    return metrics,perturb\n\ndef plots(root):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    root=Path(root);metrics,perturb=summarize(root);paths=[]\n    plt.rcParams.update({\'font.size\':10,\'axes.spines.top\':False,\'axes.spines.right\':False,\'pdf.fonttype\':42})\n    if len(metrics):\n        for task,frame in metrics.groupby(\'task\'):\n            agg=frame.groupby([\'model\',\'coordinate\']).rmse.mean().unstack()\n            fig,ax=plt.subplots(figsize=(12,6));agg.plot.bar(ax=ax,width=.82)\n            ax.set_ylabel(\'Mean participant RMSE (target units)\');ax.set_xlabel(\'Model\')\n            ax.set_title(f\'{task}: coordinate sensitivity | {frame.participant_id.nunique()} evaluated participant(s)\')\n            ax.tick_params(axis=\'x\',rotation=20);ax.legend(title=\'Coordinates\',bbox_to_anchor=(1.01,1))\n            fig.text(.02,.02,\'Descriptive results. Check participant counts and paired comparisons CSV; incomplete models/modes are omitted, never zero-filled.\',fontsize=9)\n            fig.tight_layout(rect=(0,.05,1,1));path=root/f\'{task}_coordinate_RMSE.png\';fig.savefig(path,dpi=180);fig.savefig(path.with_suffix(\'.pdf\'));plt.close(fig);paths.append(path)\n    if len(perturb) and \'translation_sd\' in perturb:\n        for (task,mode),frame in perturb.dropna(subset=[\'translation_sd\']).groupby([\'task\',\'coordinate\']):\n            fig,axes=plt.subplots(1,2,figsize=(12,5))\n            for family,part in frame.groupby(\'model\'):\n                for ax,measure in zip(axes,[\'rmse_change\',\'prediction_change_rmse\']):\n                    values=part.groupby(\'translation_sd\')[measure].mean().sort_index()\n                    values=pd.concat([values,pd.Series({0.:0.})]).sort_index()\n                    ax.plot(values.index,values.to_numpy(),marker=\'o\',label=family)\n            for ax in axes:ax.axhline(0,color=\'black\',lw=.6);ax.set_xlabel(\'Translation / outer-training input SD\');ax.grid(alpha=.15)\n            axes[0].set_ylabel(\'Change in participant RMSE (positive = worse)\')\n            axes[1].set_ylabel(\'Prediction change RMSE (target units)\')\n            axes[1].legend(fontsize=8)\n            fig.suptitle(f\'{task} / {mode}: translation sensitivity | {frame.participant_id.nunique()} participant(s)\')\n            fig.text(.02,.02,\'Fixed saved models; no refitting under perturbation. Normalized-coordinate invariance can follow directly from preprocessing.\',fontsize=9)\n            fig.tight_layout(rect=(0,.07,1,.94));path=root/f\'{task}_{mode}_translation_sensitivity.png\';fig.savefig(path,dpi=180);fig.savefig(path.with_suffix(\'.pdf\'));plt.close(fig);paths.append(path)\n    if len(perturb) and \'yaw_degrees\' in perturb:\n        for (task,mode),frame in perturb.dropna(subset=[\'yaw_degrees\']).groupby([\'task\',\'coordinate\']):\n            matrix=frame.groupby([\'model\',\'yaw_degrees\']).rmse_change.mean().unstack()\n            fig,ax=plt.subplots(figsize=(10,5));matrix.plot.bar(ax=ax);ax.axhline(0,color=\'black\',lw=.7)\n            ax.set_ylabel(\'Change in participant RMSE\');ax.set_title(f\'{task} / {mode}: coordinate-frame rotation sensitivity\')\n            ax.tick_params(axis=\'x\',rotation=20);fig.tight_layout();path=root/f\'{task}_{mode}_rotation_sensitivity.png\';fig.savefig(path,dpi=180);fig.savefig(path.with_suffix(\'.pdf\'));plt.close(fig);paths.append(path)\n    return paths\n'}
KNOWN_HEIGHTS_M = {'SUB01': 1.68, 'SUB02': 1.7, 'SUB03': 1.65, 'SUB04': 1.58, 'SUB05': 1.69, 'SUB06': 1.515, 'SUB07': 1.585, 'SUB08': 1.68, 'SUB09': 1.57, 'SUB10': 1.79, 'SUB11': 1.545, 'SUB12': 1.66, 'SUB13': 1.67, 'SUB14': 1.68, 'SUB15': 1.68, 'SUB16': 1.67, 'SUB17': 1.71, 'SUB18': 1.735, 'SUB19': 1.79, 'SUB20': 1.65, 'SUB21': 1.76, 'SUB22': 1.665, 'SUB23': 1.615, 'SUB24': 1.62, 'SUB25': 1.69, 'SUB26': 1.585}
HEIGHT_PROVENANCE = 'PDGinfo_Shida.xlsx, PDGinfo sheet, Height (cm); exact ID matching; file SHA256 7c85667796fb2b127da23e762230b7539e0fa1c1f253b0a049d2d01e6c20e229'
CODE.mkdir(parents=True,exist_ok=True)
for name,source in SOURCES.items():(CODE/name).write_text(source,encoding='utf-8')
if str(CODE) not in sys.path:sys.path.insert(0,str(CODE))
for name in SOURCES:sys.modules.pop(name[:-3],None)
import numpy as np, pandas as pd
from IPython.display import display, Image
from data import TASK_ORDER
from revision_data import legacy_cohort
from coordinate_checks import (MODES, FAMILIES, CoordinateInputs, make_templates, seed_known_heights, verify_original_source,
                               run_task, summarize, plots, record, canonical_hash)
from models import configure_tensorflow
if not np.isfinite(TRANSLATION_SD_MULTIPLIERS).all() or not np.isfinite(ROTATION_ANGLES_DEG).all():
    raise ValueError('Perturbations must be finite')
if set(REQUESTED_MODES)-set(MODES) or set(REQUESTED_MODELS)-set(FAMILIES):raise ValueError('Unknown model or coordinate mode')
tasks=TASK_ORDER if TASK_SELECTION.strip().lower()=='all' else [t.strip() for t in TASK_SELECTION.split(',')]
if set(tasks)-set(TASK_ORDER) or len(tasks)!=len(set(tasks)):raise ValueError('Unknown/duplicate task')
outers='all' if OUTER_SELECTION.strip().lower()=='all' else [s.strip() for s in OUTER_SELECTION.split(',')]
options={'translation':RUN_TRANSLATION_TESTS,'translation_sd_multipliers':TRANSLATION_SD_MULTIPLIERS,
         'rotation':RUN_ROTATION_TESTS,'yaw_degrees':ROTATION_ANGLES_DEG}
OUTPUT.mkdir(parents=True,exist_ok=True)
hardware=configure_tensorflow(reference['config']['seed'])
print('Runtime:',hardware)

## Load data, create missing-input templates, and inspect readiness

Use `PROJECT/coordinate_inputs/<TASK>/` for optional inputs, for example `.../coordinate_inputs/AP_Left/`.

- **Translation normalization:** subtract the first ankle sample of each cycle. Available immediately from the original ZIP.
- **Pelvis centering:** subtract synchronized pelvis position at each phase point; requires `pelvis_xyz` and a verified common frame.
- **Direction alignment:** translate to the cycle's start and rotate 3D coordinates into a force-independent walking frame.
- **Body scaling:** translate to the cycle's start and divide by verified participant height or leg length. This can run on the original scalar input once the metadata is complete.
- **Combined:** pelvis centering, direction alignment and body scaling together.

The cell creates `cycle_id_reference.csv`, `participant_body_scale.csv`, `coordinate_metadata.json`, and `INPUT_GUIDE.txt`. It fills a fresh, blank body-scale template using the **Height (cm)** values in your supplied **PDGinfo_Shida.xlsx**, converted to metres and matched by participant ID. It does not replace existing user-entered body scales. The original ankle-coordinate unit still requires verification: set `CONFIRMED_ANKLE_UNITS` and document `ANKLE_UNITS_EVIDENCE` in the first configuration cell, or complete the metadata JSON.

Supply real synchronized arrays in `coordinate_inputs.npz`, following the guide. Do not substitute zero pelvis positions, guessed directions or estimated anthropometry. After adding verified data, use a new RUN_ID to keep the analysis versions distinct.

Alignment and rotation additionally require confirmation that the unchanged GRF targets are already defined in the walking frame. If they are laboratory-frame forces, first reconstruct a consistently harmonized 3D cohort; this companion does not guess force transformations.

In [ ]:
cohorts={}; readiness=[]
for task in tasks:
    print('Loading',task,flush=True)
    cohort=legacy_cohort(DATA_ZIP,task);cohorts[task]=cohort
    source=SOURCE_ROOT/task/'absolute'
    if (source/'prespecification.json').exists():verify_original_source(source,cohort,CODE)
    folder=SUPPLEMENT/task;make_templates(cohort,folder)
    seed_known_heights(folder,KNOWN_HEIGHTS_M,HEIGHT_PROVENANCE,CONFIRMED_ANKLE_UNITS,ANKLE_UNITS_EVIDENCE)
    inputs=CoordinateInputs(cohort,folder)
    for mode in REQUESTED_MODES:
        try:inputs.require(mode);status,reason='ready',''
        except ValueError as error:status,reason='blocked',str(error)
        readiness.append({'task':task,'coordinate':mode,'status':status,'reason':reason})
    print('Participants:',len(cohort.ds.subjects),'cycles:',len(cohort.ds.x),'input guide:',folder/'INPUT_GUIDE.txt')
ready=pd.DataFrame(readiness);display(ready)
ready.to_csv(OUTPUT/'input_readiness.csv',index=False)
print('Missing modes remain explicitly blocked; available modes will still run.')
if STRATEGY=='frozen_selection':
    print('FROZEN SENSITIVITY MODE: each available new coordinate requires one refit per completed model, not a new hyperparameter search.')
else:
    print('FULL INNER RESELECTION MODE: each candidate runs across all inner participants. This may require substantial GPU time.')

## Run the coordinate analyses and perturbation tests

The absolute-coordinate models are evaluated first. **Translation perturbations are inference only:** fixed positive/negative offsets are expressed as multiples of the standard deviation of the outer-training ankle samples. No test-participant values are used to set the offsets. They are not labelled as metres because physical units have not been verified. Each shift is applied consistently to ankle and pelvis coordinates where present.

New coordinate representations require fitting new weights. In the default fast mode, each model retains its original inner-selected hyperparameters and fixed epoch count. In `nested_reselect`, selection and early stopping use separate whole participants for each candidate, followed by an outer refit with the median inner best epoch, rounded upward. Outer test cycles are never used for selection or early stopping.

Every completed case and perturbation is saved to Drive. Restarting and using the **same RUN_ID, strategy and settings** reuses valid completed cases and inner-fold caches. A fit interrupted before saving may need to restart. Changing data, software or analysis settings requires a new RUN_ID. Run only one writer per output experiment at a time.

Each model is an individual comparison. There is no ensemble, no external testing, and no model winner is selected from the displayed outer-test results.

In [ ]:
for task,cohort in cohorts.items():
    source=SOURCE_ROOT/task/'absolute'
    spec=json.loads((source/'prespecification.json').read_text()) if (source/'prespecification.json').exists() else reference
    cfg=dict(spec['config']);cfg['smoke']=False
    print('START',task,STRATEGY,flush=True)
    try:
        run_task(cohort,source,OUTPUT,SUPPLEMENT/task,outers,REQUESTED_MODES,REQUESTED_MODELS,
                 STRATEGY,options,cfg)
    except KeyboardInterrupt:
        print('Interrupted. Completed cases remain on Drive. Rerun with identical settings to resume.');raise
    except Exception as error:
        record(OUTPUT,f'{task}/task_setup','failed',str(error))
print('Requested execution finished. Check the status table: finished execution does not mean every requested analysis completed.')

## Results and plots

MAE/RMSE/NRMSE are calculated per cycle and averaged within the participant. NRMSE divides each cycle's RMSE by its measured force range. Paired coordinate comparisons use the same participant and model; positive `rmse_improvement` favors the normalized representation. Perturbation `rmse_change` is perturbed minus unperturbed error; positive values indicate deterioration. `prediction_change_rmse` measures how much the predictions changed, independently of the target error.

With SUB01 alone, all results are descriptive; the seven cycles are not seven independent participants. The summary includes participant counts and does not manufacture confidence intervals. If extending to the full cohort, use paired participant-level comparisons and report missing/failed cases.

Translation invariance of centered inputs, and rotation invariance after appropriate direction alignment, can follow mathematically from preprocessing. Such invariance is not proof that the learned mapping is biomechanical. Frozen-selection results and fully reselected nested results must be described separately.

In [ ]:
coordinate_metrics,perturbation_metrics=summarize(OUTPUT)
if (OUTPUT/'status.csv').exists():display(pd.read_csv(OUTPUT/'status.csv'))
if len(coordinate_metrics):
    display(coordinate_metrics)
    if (OUTPUT/'paired_coordinate_comparisons.csv').exists():display(pd.read_csv(OUTPUT/'paired_coordinate_comparisons.csv'))
else:print('No coordinate model has completed. Inspect status.csv for specific reasons.')
if len(perturbation_metrics):display(perturbation_metrics)
for path in plots(OUTPUT):display(Image(filename=str(path)))
print('Persistent results:',OUTPUT)

In [ ]:
# Export reports and plots, without large trained weights or tuner histories.
import zipfile,hashlib
from google.colab import files
EXPORT_PREDICTIONS = True
DOWNLOAD_REPORT = True
manifest={'source_experiment':SOURCE_EXPERIMENT,'run_id':RUN_ID,'strategy':STRATEGY,
          'tasks':tasks,'outer_selection':outers,'modes_requested':REQUESTED_MODES,
          'models_requested':REQUESTED_MODELS,'perturbations':options,
          'source_hashes':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
          'notes':['No external data evaluated','Blocked/failed cases are not completed results',
                   'Frozen selection is a sensitivity refit, not coordinate-specific inner reselection']}
(OUTPUT/'notebook_manifest.json').write_text(json.dumps(manifest,indent=2))
archive=Path('/content')/f'{RUN_ID}_{STRATEGY}_reports.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.iterdir():
        if path.is_file() and path.suffix in ('.csv','.json','.png','.pdf','.txt'):z.write(path,path.name)
    for path in (OUTPUT/'cases').rglob('*'):
        if not path.is_file() or any(x in ('tuner','candidates','refit') for x in path.relative_to(OUTPUT).parts):continue
        allowed=path.name in ('case_manifest.json','complete.json','participant_metrics.csv','cycle_metrics.csv','selected.json')
        if allowed or (EXPORT_PREDICTIONS and path.name=='predictions.npz'):z.write(path,str(path.relative_to(OUTPUT)))
    for name,source in SOURCES.items():z.writestr('analysis_code/'+name,source)
print('Report archive:',archive)
if DOWNLOAD_REPORT:files.download(str(archive))